# MI cascade analysis in HGSOC

See [README.md](README.md) for the execution order, upstream inputs, commands and paper mapping.
Run the unified entry point to save executed copies and local outputs. Scientific variants and their parameters are retained below.


## 1. Imports and device setup

The wildcard imports supply pd and torch through the local SpiderNet modules. Additional plotting and analysis imports appear in later sections.


In [ ]:
import json

import numpy as np

from SpiderNet.utils import *
from SpiderNet.config import *
from dataclasses import fields
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

cuda_available = torch.cuda.is_available()
if cuda_available:
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

## 2. Configure data and result paths

The paths below point to the existing HGSOC data layout. `HGSOC_modeltraining_setup.json`, when present under the initial `OUTPUT_ROOT`, overrides `PROCESSED_DATA_DIR` and `OUTPUT_ROOT`; it does not override `DATA_ROOT`. The subsequent `run_dirs.json` supplies the analysis output directory. These external data locations must be available before execution.


In [ ]:
from workflow_paths import (PROCESSED_DATA_DIR, RESULTS_ROOT, DATA_ROOT, OUTPUT_ROOT, input_path, output_path, saved, ensure_output)
ensure_output()


## 3. Locate the trained run

The training workflow writes `run_dirs.json`. The run directory name must contain `Result_dim<N>` because the MI dimension is parsed from that name.


In [ ]:
import re
from workflow_paths import run_dirs as _configured_run_dirs
run_dirs = dict(_configured_run_dirs)
run_dir = Path(run_dirs["run_dir"])
match = re.search(r"Result_dim(\d+)", str(run_dir))
if match is None:
    raise ValueError(f"Cannot parse dim_envir from run directory: {run_dir}")
dim_envir = int(match.group(1))
print(run_dirs)


## 4. Load processed HGSOC data

The preprocessing notebook `spidernet_dataloading_MIdimselection_HGSOC.ipynb` supplies the processed bundle. This analysis uses sample-aligned AnnData objects and graph objects with directed `edge_index` and `cellpair_LRpair_neigh` matrices. AnnData requires `cell.types`, samples, and gene names; cell, gene, LR-pair, and sample ordering must match the trained factors and metadata.


In [ ]:
from SpiderNet.io import load_processed_data
processed = load_processed_data(PROCESSED_DATA_DIR)

SpiderNet_data_pyg_list = processed.spidernet_data
adata_list = processed.adata_list
samples_name_list = [np.unique(adata_list[i].obs["samples"])[0] for i in range(len(adata_list))]

## 5. Load inferred MI activities

`Factor_envir_list.pkl` contains one directed-edge-by-MI matrix per sample, aligned with the processed graph edge order.


In [ ]:
Factor_envir_list = pd.read_pickle(input_path(run_dirs["run_dir"] + f"/Factor_envir_list.pkl"))

## 6. Screen ordered MI cascade pairs

`MI_colocalization_analysis` performs normalization, cell-type-pair-constrained permutations, pooled count comparison, and Benjamini-Hochberg correction. No permutation seed is set in this notebook. The returned field named `colocal_count_merge_sum_zscore` is a stabilized log2 observed/null count ratio, not a conventional z-score.


In [ ]:
# Global screen: retain globally maximum-normalized MI activity > MI_threshold.
MI_threshold = 0.6

In [ ]:
# The count-score cutoff is log2(1.4); the variable retains its historical name.
zscore_countcolocal_threshold = 1.4
pvalue_adjusted_threshold = 0.001

In [ ]:
from SpiderNet.analysis import MI_colocalization_analysis
result = MI_colocalization_analysis(
    Factor_envir_list=Factor_envir_list,
    SpiderNet_data_pyg_list=SpiderNet_data_pyg_list,
    adata_list=adata_list,
    MI_threshold=MI_threshold,
    nperm=100,
    fdr_alpha=0.05,
    celltype_col="cell.types",
    progress_every=5,
    zscore_countcolocal_threshold = zscore_countcolocal_threshold,
    pvalue_adjusted_threshold = pvalue_adjusted_threshold
)
colocal_count_merge_sum_zscore = result["colocal_count_merge_sum_zscore"]
colocal_count_merge_sum_pvalue = result["colocal_count_merge_sum_pvalue_fdr"]

In [ ]:
MI_colocal_summary = pd.DataFrame({
    "zscore_countcolocal": np.array(colocal_count_merge_sum_zscore).flatten(),
    "pvalue_adjusted": np.array(colocal_count_merge_sum_pvalue).flatten(),
    "MI_first": np.repeat(np.arange(1, dim_envir + 1), dim_envir),
    "MI_second": np.tile(np.arange(1, dim_envir + 1), dim_envir)
})
# The selection below uses an explicit adjusted-P cutoff of 0.001.
MI_colocal_summary_significant = MI_colocal_summary.loc[(MI_colocal_summary['zscore_countcolocal'] > np.log2(zscore_countcolocal_threshold)) & (MI_colocal_summary['pvalue_adjusted'] < 0.001),:]

In [ ]:
# Convert a log2 count score to its count ratio for inspection.
np.power(2,0.481)

## 7. Cluster the log2 cascade count ratios

Rows are upstream MIs and columns are downstream MIs. Yellow outlines apply the same count-score and adjusted-P cutoffs as the selection above.


In [ ]:
# =========================================
# Hierarchical clustering heatmap
# =========================================
g = sns.clustermap(
    colocal_count_merge_sum_zscore,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,                   # displayed color limits
    figsize=(7, 6),
    square=False,                      # rectangular heatmap cells
    dendrogram_ratio=(0.1, 0.1),
    cbar_pos=None,                     # no color bar in this output
)

# =========================================
# Retrieve clustered row/column order
# =========================================
row_order = g.dendrogram_row.reordered_ind
col_order = g.dendrogram_col.reordered_ind

# =========================================
# Highlight significant MI-MI pairs 
# (adjusted P < 1e-3 and log2 count score > log2(zscore_countcolocal_threshold))
# =========================================
for i_new, i_old in enumerate(row_order):
    for j_new, j_old in enumerate(col_order):
        p_val = colocal_count_merge_sum_pvalue.iloc[i_old, j_old]
        z_val = colocal_count_merge_sum_zscore.iloc[i_old, j_old]
        if p_val < 1e-3 and z_val > np.log2(zscore_countcolocal_threshold):
            g.ax_heatmap.add_patch(
                Rectangle(
                    (j_new, i_new),
                    1, 1,
                    fill=False,
                    edgecolor="yellow",
                    lw=2.5
                )
            )

# =========================================
# Axis labels and tick style
# =========================================
g.ax_heatmap.set_xlabel("Second-layer meta-interaction", fontsize=15, labelpad=12)
g.ax_heatmap.set_ylabel("First-layer meta-interaction", fontsize=15, labelpad=12)

# Adjust tick labels orientation and font
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45, ha="right", fontsize=11
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0, fontsize=11
)

# =========================================
# Layout and save
# =========================================
g.fig.tight_layout()
g.fig.subplots_adjust(bottom=0.18, left=0.18)   # add margins to avoid label clipping

out_path = run_dirs["run_dir"] + "/Heatmap_Zscore_Colocalization_Count_allsample.png"
g.fig.savefig(out_path, dpi=300)
out_path_pdf = run_dirs["run_dir"] + "/Heatmap_Zscore_Colocalization_Count_allsample.pdf"
g.fig.savefig(out_path_pdf, dpi=300)
plt.show()
plt.close()

print(f"✅ Heatmap saved to: {out_path}")


## 8. Summarize cell-type triplet proportions

The library pools triplet counts across samples and divides by the total count for each MI pair. Triplet types are retained when their maximum proportion across selected MI pairs exceeds 0.1. The current call passes the loaded factors directly; the global screen above normalizes its own copy. The two heatmaps show all retained triplet types and the subset containing malignant cells.


In [ ]:
from SpiderNet.analysis import summarize_mi_cascade_celltype_triples
cascade_res = summarize_mi_cascade_celltype_triples(
    MI_colocal_summary_significant=MI_colocal_summary_significant,
    Factor_envir_norm_list=Factor_envir_list,
    hyper_edge_adj_list=result["hyper_edge_adj_list"],
    SpiderNet_data_pyg_list=SpiderNet_data_pyg_list,
    adata_list=adata_list,
    MI_threshold=MI_threshold,
    celltype_col="cell.types",
    sample_col="samples",
    select_triple_prop_threshold=0.1,
    progress_every=10,
)
prop_avg_allsample_filter_merge_pivot_select = cascade_res["prop_avg_allsample_filter_merge_pivot_select"]

In [ ]:
from SpiderNet.visualization import Heatmap_celltypetriplet_prop
# ==============================================================
# 1. Heatmap for all celltype triples across all MI cascades
# ==============================================================
Heatmap_celltypetriplet_prop(
    prop_avg_allsample_filter_merge_pivot_select,
    title="Average Proportion of Celltype Triples across MI Cascades",
    xlabel="Celltype Triple",
    ylabel="MI Cascade",
    filename="Heatmap_Average_prop_Celltype_triple_across_MI_cascades.pdf",
    highlight_malignant=True,
    file_savepath_main=run_dirs["run_dir"]
)

In [ ]:
# ==============================================================
# 2. Heatmap for Malignant-related celltype triples only
# ==============================================================
prop_malignant = prop_avg_allsample_filter_merge_pivot_select.loc[
    prop_avg_allsample_filter_merge_pivot_select.index.str.contains("Malignant")
]

Heatmap_celltypetriplet_prop(
    prop_malignant,
    title="Malignant-related Celltype Triples across Significant MI Cascades",
    xlabel="Celltype Triple (Malignant-related)",
    ylabel="Significant MI Cascade",
    filename="Heatmap_Average_prop_Celltype_triple_across_MI_cascades_Malignant.pdf",
    file_savepath_main = run_dirs["run_dir"]
)

## 9. Compare the selected upstream-gated neighborhoods

Load the malignant subtype assignments supplied by `HGSOC_Malignantsubtype_analysis_V2.ipynb`. The loader first uses an available AnnData file with Scanpy, then falls back to a cluster-assignment CSV. This dependency is required even with `REQUIRE_CT3_MALIGNANT_C5 = False`; that setting controls triplet filtering, not assignment loading. The exported C5 barcode list records the available assignment set.


In [ ]:
# ==============================================================
# Load the malignant C5 assignment from the malignant-subtype workflow.
# --------------------------------------------------------------
# The malignant-subtype workflow supplies:
#   run_dir/adata_choose_Malignant.h5ad
# and stores the malignant subtype assignment in:
#   adata_choose.obs["MI_louvain"] and adata_choose.obs["Malignant_C5"]
#
# From this point on, downstream analyses may restrict the third cell
# in any ... -> ... -> Malignant triplet to Malignant_C5 cells.
# ==============================================================

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import scanpy as sc
except ImportError:
    sc = None

# REQUIRE_CT3_MALIGNANT_C5 = True
REQUIRE_CT3_MALIGNANT_C5 = False
MALIGNANT_C5_LABEL = "Malignant_C5"
MALIGNANT_C5_CLUSTER_NUMBER = "5"
MALIGNANT_C5_CLUSTER_LABEL = "C5"

_malignant_choose_candidates = [
    Path(run_dirs["run_dir"]) / "adata_choose_Malignant.h5ad",
    Path(run_dirs["run_dir"]) / "adata_choose_Malignant_updated.h5ad",
    PROCESSED_DATA_DIR / "adata_choose_Malignant.h5ad",
]

_malignant_cluster_csv_candidates = [
    Path(run_dirs["run_dir"]) / "Cluster_assignment_Malignant_barcode.csv",
    Path(run_dirs["run_dir"]) / "Cluster_assignment_Malignant.csv",
]

adata_choose_malignant_path = next((p for p in _malignant_choose_candidates if input_path(p).exists()), None)
cluster_csv_path = next((p for p in _malignant_cluster_csv_candidates if input_path(p).exists()), None)

if adata_choose_malignant_path is not None and sc is not None:
    adata_choose = sc.read_h5ad(input_path(adata_choose_malignant_path))

    if "barcode" in adata_choose.obs.columns:
        _malignant_barcode_series = adata_choose.obs["barcode"].astype(str)
    else:
        _malignant_barcode_series = pd.Series(
            adata_choose.obs_names.astype(str),
            index=adata_choose.obs_names,
        )

    if "Malignant_C5" in adata_choose.obs.columns:
        _is_c5 = adata_choose.obs["Malignant_C5"].astype(str).eq("C5").to_numpy()
        _c5_source = f"{adata_choose_malignant_path}::obs['Malignant_C5'] == 'C5'"
    elif "MI_louvain" in adata_choose.obs.columns:
        _cluster = adata_choose.obs["MI_louvain"].astype(str).str.replace(r"^C", "", regex=True)
        _is_c5 = _cluster.eq(MALIGNANT_C5_CLUSTER_NUMBER).to_numpy()
        _c5_source = f"{adata_choose_malignant_path}::obs['MI_louvain'] == '5'"
    else:
        raise KeyError(
            "Cannot find Malignant_C5 or MI_louvain in adata_choose.obs. "
            f"Available columns: {list(adata_choose.obs.columns)}"
        )

    malignant_c5_barcodes = _malignant_barcode_series.loc[_is_c5].astype(str).tolist()

elif cluster_csv_path is not None:
    cluster_assign_df = pd.read_csv(input_path(cluster_csv_path), index_col=0)

    if "MI_louvain_C" in cluster_assign_df.columns:
        _is_c5 = cluster_assign_df["MI_louvain_C"].astype(str).eq(MALIGNANT_C5_CLUSTER_LABEL)
    elif "MI_louvain" in cluster_assign_df.columns:
        _cluster = cluster_assign_df["MI_louvain"].astype(str).str.replace(r"^C", "", regex=True)
        _is_c5 = _cluster.eq(MALIGNANT_C5_CLUSTER_NUMBER)
    else:
        raise KeyError(
            "Cannot find MI_louvain_C or MI_louvain in the malignant cluster assignment CSV. "
            f"Available columns: {list(cluster_assign_df.columns)}"
        )

    malignant_c5_barcodes = cluster_assign_df.index.astype(str)[_is_c5.to_numpy()].tolist()
    _c5_source = str(cluster_csv_path)

else:
    raise FileNotFoundError(
        "Cannot find Malignant_C5 assignment from HGSOC_Malignantsubtype_analysis.\n"
        "Expected one of:\n"
        + "\n".join(f"  - {p}" for p in _malignant_choose_candidates + _malignant_cluster_csv_candidates)
    )

MALIGNANT_C5_BARCODE_SET = frozenset(map(str, malignant_c5_barcodes))

if len(MALIGNANT_C5_BARCODE_SET) == 0:
    raise ValueError("No Malignant_C5 barcodes were found. Please check HGSOC_Malignantsubtype_analysis output.")

# Export the loaded C5 barcode inventory, including when C5 filtering is disabled.
malignant_c5_barcode_path = Path(run_dirs["run_dir"]) / "Malignant_C5_barcodes_used_for_MIcascade.csv"
pd.DataFrame({"barcode": sorted(MALIGNANT_C5_BARCODE_SET)}).to_csv(malignant_c5_barcode_path, index=False)


def _barcode_array_from_adata(adata):
    """Return one barcode per row of an AnnData object."""
    for col in ("barcode", "Barcode", "cell_id", "cell", "CellID"):
        if col in adata.obs.columns:
            return adata.obs[col].astype(str).to_numpy()
    return adata.obs_names.astype(str).to_numpy()


def _malignant_c5_node_mask_for_adata(adata):
    """Boolean row mask for Malignant_C5 cells in a sample-level AnnData."""
    return np.isin(_barcode_array_from_adata(adata), list(MALIGNANT_C5_BARCODE_SET))


def _filter_triplets_to_malignant_c5_if_needed(
    triplets,
    adata,
    sample=None,
    cascade_definition=None,
    cell3_position=2,
    group=None,
    context="",
):
    """
    Optionally retain triplets whose third cell is assigned to Malignant_C5.

    With REQUIRE_CT3_MALIGNANT_C5=False, return the input triplets unchanged.

    This intentionally keeps cascade_definition.cell_types as
    (..., ..., "Malignant") because adata.obs["cell.types"] stores the broad
    cell type. The subtype restriction is applied after extracting malignant
    triplets.
    """
    triplets = np.asarray(triplets, dtype=int)

    if triplets.size == 0:
        return triplets

    if not globals().get("REQUIRE_CT3_MALIGNANT_C5", False):
        return triplets

    # Only apply the subtype filter when the third broad cell type is Malignant.
    if cascade_definition is not None:
        try:
            expected_ct3 = str(cascade_definition.cell_types[cell3_position])
            if expected_ct3 != "Malignant":
                return triplets
        except Exception:
            pass

    barcode_arr = _barcode_array_from_adata(adata)
    cell3_nodes = triplets[:, cell3_position].astype(int)

    valid = (cell3_nodes >= 0) & (cell3_nodes < len(barcode_arr))
    keep = np.zeros(triplets.shape[0], dtype=bool)
    keep[valid] = np.isin(barcode_arr[cell3_nodes[valid]].astype(str), list(MALIGNANT_C5_BARCODE_SET))

    return triplets[keep]


_malignant_c5_counts_by_sample = []
for _i, _adata in enumerate(adata_list):
    _mask = _malignant_c5_node_mask_for_adata(_adata)
    _sample_name = samples_name_list[_i] if "samples_name_list" in globals() and _i < len(samples_name_list) else str(_i)
    _malignant_c5_counts_by_sample.append({
        "sample_index": _i,
        "sample_name": _sample_name,
        "n_Malignant_C5_cells": int(np.sum(_mask)),
    })

malignant_c5_counts_by_sample_df = pd.DataFrame(_malignant_c5_counts_by_sample)

print(f"Loaded {len(MALIGNANT_C5_BARCODE_SET):,} Malignant_C5 barcodes from: {_c5_source}")
print(f"Saved barcode list to: {malignant_c5_barcode_path}")
display(malignant_c5_counts_by_sample_df.head())
print(
    "From Section 9 onward, if ct3 is Malignant and REQUIRE_CT3_MALIGNANT_C5=True, "
    "triplets are restricted to cell3=Malignant_C5."
)


In [ ]:
print(MI_colocal_summary_significant)

### 9.1. Select MIs, cell types, and comparison settings

MI identifiers are one-based here and converted to zero-based matrix indices below. Commented settings retain alternative cascade definitions and scoring choices.


In [ ]:

# ==============================================================
# Upstream-gated triplet analysis settings
# --------------------------------------------------------------
# Comparison groups:
#   Group 1: cell1 --(MI-up interacting + spatial neighboring)--> cell2 --(spatial neighboring)--> cell3
#   Group 2: cell1 --(MI-up NOT interacting + spatial neighboring)--> cell2 --(spatial neighboring)--> cell3
#
# The downstream edge is required only to be a spatial neighbor.
# MI-down is NOT used for grouping; it is measured as a feature.
# ==============================================================

# ----- User-editable cascade of interest -----
MI_first_of_interest = 12
MI_second_of_interest = 10
celltype_triple_of_interest = "Monocyte -> Fibroblast -> Malignant"
# celltype_triple_of_interest = "Malignant -> Fibroblast -> Malignant"

# MI_first_of_interest = 6
# MI_second_of_interest = 10
# celltype_triple_of_interest = "Fibroblast -> Fibroblast -> Malignant"

# ----- Upstream MI gate thresholds -----
# MI-up interacting:     MI_up_strength >= MI_UP_THRESHOLD_HIGH
# MI-up not interacting: MI_up_strength <  MI_UP_THRESHOLD_LOW
# Candidate middle range is excluded from the two comparison groups.
# MI_UP_THRESHOLD_LOW = 0.50
# MI_UP_THRESHOLD_LOW = 0.05
# MI_UP_THRESHOLD_HIGH = 0.2
# MI_UP_THRESHOLD_LOW = 0.10
MI_UP_THRESHOLD_LOW = 1e-3
# MI_UP_THRESHOLD_LOW = 0.05
# MI_UP_THRESHOLD_LOW = 0.50
MI_UP_THRESHOLD_HIGH = 0.50

# ----- Module-score convention -----
# Available options:
#   "mean":                 direct average expression across genes.
#   "zscore_mean":          gene-wise z-score within each sample-level reference cell population,
#                           then average genes.
#   "zscore_mean_global":   concatenate the relevant reference cell population across all slices,
#                           compute one global mean/std for each gene, z-score cells in each slice
#                           using those global statistics, then average genes.
#   "addmodule":            Scanpy AddModule-style score using sc.tl.score_genes.
#                           Scores are computed within each slice/sample separately, then
#                           extracted for the selected Cell2/Cell3 nodes.
#
# MODULE_SCORE_METHOD = "mean"
# MODULE_SCORE_METHOD = "zscore_mean"
MODULE_SCORE_METHOD = "zscore_mean_global"
# MODULE_SCORE_METHOD = "addmodule"

# Settings used only when MODULE_SCORE_METHOD = "addmodule".
ADD_MODULE_CTRL_SIZE = 50
ADD_MODULE_RANDOM_STATE = 0
ADD_MODULE_USE_RAW = False

# ----- Sample-level test -----
# True: the statistics table tests matched sample means with paired Wilcoxon.
# False: the statistics table uses an unpaired Mann-Whitney U test.
# The plotting helper independently annotates an unpaired test in both modes.
USE_PAIRED_SAMPLE_TEST = True

# ----- Optional deduplication -----
# With True, repeated triplets contribute each edge/cell unit once per group and feature.
DEDUPLICATE_UNITS_WITHIN_GROUP = True

# ----- Cross-group overlap exclusion -----
# To prevent reuse of cells and edges across groups, remove MI-up-negative triplets
# if they reuse any edge or any cell already present in the MI-up-positive group
# within the same sample.
#
# Edge overlap is checked using both spatial edges in the triplet:
#   edge12 = cell1 -> cell2 and edge23 = cell2 -> cell3.
# Cell overlap is checked using the listed triplet positions:
#   0 = cell1, 1 = cell2, 2 = cell3.
# Use (1, 2) if you only want to exclude overlap in the scored cells.
EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS = True
EXCLUDE_OVERLAP_EDGE_ROLES = ("upstream", "downstream")
EXCLUDE_OVERLAP_CELL_POSITIONS = (0, 1, 2)


# ----- Edge validation -----
# Every retained triplet must map to two directed spatial edges in sample.edge_index:
#   edge12 = cell1 -> cell2 and edge23 = cell2 -> cell3.
# A small number of missing edge mappings can be dropped defensively.
# If the missing fraction exceeds this threshold, the notebook stops because
# that usually indicates an edge-direction or triplet-extraction mismatch.
MAX_TRIPLET_EDGE_DROP_FRACTION = 0.05
RAISE_ON_HIGH_EDGE_MAPPING_DROPOUT = True


# ----- Cell3 subtype restriction -----
# Retain the previously configured C5 restriction; the default above is False.
REQUIRE_CT3_MALIGNANT_C5 = globals().get("REQUIRE_CT3_MALIGNANT_C5", True)

# ----- Groups used throughout this analysis -----
GROUP_UP_POS = "MIup_interacting"
GROUP_UP_NEG = "MIup_not_interacting"
UPSTREAM_GATED_GROUPS = (GROUP_UP_POS, GROUP_UP_NEG)

PRETTY_GROUP_LABELS_UPSTREAM_GATED = {
    GROUP_UP_POS: "MI-up+\nneighbor",
    GROUP_UP_NEG: "MI-up-\nneighbor",
}

GROUP_COLORS_UPSTREAM_GATED = {
    GROUP_UP_POS: "#3d91ce",
    GROUP_UP_NEG: "#81b6de",
}

# ----- Pathway-level LR coexpression to score -----
# Resolve these pathway names against LR_meta_incellchatdb.csv.
UPSTREAM_PATHWAY_NAME = "SPP1_upstream"
UPSTREAM_PATHWAY_NAMES = ["SPP1"]

# UPSTREAM_PATHWAY_NAME = "VTN_upstream"
# UPSTREAM_PATHWAY_NAMES = ["VTN"]

DOWNSTREAM_PATHWAY_NAME = "THBS_downstream"
DOWNSTREAM_PATHWAY_NAMES = ["THBS"]

print("Upstream-gated comparison groups:")
print(f"  {GROUP_UP_POS}: MI-{MI_first_of_interest} >= {MI_UP_THRESHOLD_HIGH} on cell1->cell2")
print(f"  {GROUP_UP_NEG}: MI-{MI_first_of_interest} <  {MI_UP_THRESHOLD_LOW} on cell1->cell2")
print("Downstream cell2->cell3 is required only to be spatially neighboring.")
print(f"MI-down MI-{MI_second_of_interest} will be measured as a feature, not used for grouping.")
print(f"MODULE_SCORE_METHOD = {MODULE_SCORE_METHOD}")
_module_score_method_norm = str(MODULE_SCORE_METHOD).lower().replace("-", "_")
if _module_score_method_norm in {"addmodule", "add_module", "scanpy_score_genes", "score_genes"}:
    print(
        "AddModule settings: "
        f"ctrl_size={ADD_MODULE_CTRL_SIZE}, "
        f"random_state={ADD_MODULE_RANDOM_STATE}, "
        f"use_raw={ADD_MODULE_USE_RAW}"
    )
elif _module_score_method_norm in {"zscore_mean_global", "global_zscore_mean", "zscore_global", "global_zscore"}:
    print(
        "Global z-score settings: gene mean/std will be computed from the matching "
        "reference cell population concatenated across all slices, then reused for "
        "each slice."
    )
print(f"USE_PAIRED_SAMPLE_TEST = {USE_PAIRED_SAMPLE_TEST}")

print(f"EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS = {EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS}")
print(f"EXCLUDE_OVERLAP_EDGE_ROLES = {EXCLUDE_OVERLAP_EDGE_ROLES}")
print(f"EXCLUDE_OVERLAP_CELL_POSITIONS = {EXCLUDE_OVERLAP_CELL_POSITIONS}")


In [ ]:

# ==============================================================
# Resolve selected MI indices, broad cell-type triple, output suffix
# ==============================================================

import os
import re
import math
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import mannwhitneyu, wilcoxon

from SpiderNet.analysis import (
    AnalysisConfig,
    CascadeDefinition,
    MICascadeAnalyzer,
    PathwaySpec,
)
import SpiderNet.analysis as spna

output_dir = run_dirs["run_dir"]
Path(output_dir).mkdir(parents=True, exist_ok=True)

MI_first_index = int(MI_first_of_interest) - 1
MI_second_index = int(MI_second_of_interest) - 1

ct1_cur = celltype_triple_of_interest.split("->")[0].strip()
ct2_cur = celltype_triple_of_interest.split("->")[1].strip()
ct3_cur = celltype_triple_of_interest.split("->")[2].strip()
cell_types_triplet = (ct1_cur, ct2_cur, ct3_cur)

ct3_subtype_cur = (
    "Malignant_C5"
    if REQUIRE_CT3_MALIGNANT_C5 and ct3_cur == "Malignant"
    else ct3_cur
)

cascade_output_suffix = (
    f"UpstreamGated_MI{MI_first_of_interest}_MI{MI_second_of_interest}_"
    f"{ct1_cur}-{ct2_cur}-{ct3_subtype_cur}"
)

print(f"Selected broad triplet: {ct1_cur} -> {ct2_cur} -> {ct3_cur}")
print(f"Cell3 subtype shown in outputs: {ct3_subtype_cur}")
print(f"MI-up index:   MI-{MI_first_of_interest} / zero-based {MI_first_index}")
print(f"MI-down index: MI-{MI_second_of_interest} / zero-based {MI_second_index}")
print(f"Output suffix: {cascade_output_suffix}")


### 9.2. Matrix access, scoring, statistics, and triplet helpers

Helpers preserve directed edge alignment, score gene sets, validate triplets, and exclude shared units across groups. Rerun this cell to reset the score caches before recomputing scores after changing inputs or reference populations. The commented plotting implementation remains as an alternative figure definition.


In [ ]:

# ==============================================================
# Helper functions: matrix access, scoring, plotting, and statistics
# ==============================================================

def _to_numpy_array(x):
    """Convert numpy / scipy sparse / torch tensor / pandas object to numpy array."""
    if sparse.issparse(x):
        return x.toarray()
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    if hasattr(x, "cpu") and hasattr(x, "numpy"):
        return x.cpu().numpy()
    return np.asarray(x)


def _matrix_subset_to_numpy(matrix, rows, cols):
    rows = np.asarray(rows, dtype=int)
    cols = np.asarray(cols, dtype=int)

    if rows.size == 0:
        return np.empty((0, len(cols)), dtype=float)
    if cols.size == 0:
        return np.empty((len(rows), 0), dtype=float)

    sub = matrix[rows, :][:, cols]
    sub = _to_numpy_array(sub)

    if sub.ndim == 1:
        sub = sub.reshape(-1, 1)

    return sub.astype(float, copy=False)


def _sample_name_from_adata(adata, sample_index):
    if "samples_name_list" in globals() and sample_index < len(samples_name_list):
        return str(samples_name_list[sample_index])
    for key in ("samples", "sample", "sample_id", "patient", "patient_id"):
        if key in adata.obs:
            vals = pd.unique(adata.obs[key].astype(str))
            if len(vals) > 0:
                return str(vals[0])
    return str(sample_index)


def _edge_array_from_edge_index(edge_index):
    """Return directed edges as an n_edges × 2 integer array.

    The returned row number is the edge id used by factor_matrix and lr_matrix.
    """
    edge_index_np = _to_numpy_array(edge_index)
    if edge_index_np.ndim != 2:
        raise ValueError(f"edge_index should be 2D; got shape {edge_index_np.shape}")

    # PyG convention: 2 × n_edges.
    if edge_index_np.shape[0] == 2:
        edges = edge_index_np.T
    # Alternative convention: n_edges × 2.
    elif edge_index_np.shape[1] == 2:
        edges = edge_index_np
    else:
        raise ValueError(
            "Cannot interpret edge_index as 2×n_edges or n_edges×2. "
            f"Got shape={edge_index_np.shape}"
        )

    if edges.shape[1] < 2:
        raise ValueError(f"edge_index converted to invalid edge array shape={edges.shape}")

    return edges[:, :2].astype(int, copy=False)


def _get_edge_count(edge_index, edge_lookup=None):
    """Return number of directed edges.

    Important: use edge_index rather than len(edge_lookup), because factor_matrix
    and lr_matrix are aligned to the row order of edge_index.  A lookup may drop
    duplicate directed pairs, so len(edge_lookup) can be smaller than n_edges.
    """
    return int(_edge_array_from_edge_index(edge_index).shape[0])


def _prepare_factor_matrix_for_edges(factor_arr, n_edges):
    """Return factor matrix with shape n_edges × n_MIs."""
    factor_np = _to_numpy_array(factor_arr).astype(float, copy=False)

    if factor_np.ndim != 2:
        raise ValueError(f"factor_arr should be 2D; got shape {factor_np.shape}")

    if factor_np.shape[0] == n_edges:
        return factor_np
    if factor_np.shape[1] == n_edges:
        return factor_np.T

    raise ValueError(
        "Cannot align factor matrix to edge_index. "
        f"factor_arr shape={factor_np.shape}, n_edges={n_edges}"
    )


def _values_by_edge_and_mi(factor_matrix, edge_ids, mi_index):
    edge_ids = np.asarray(edge_ids)
    out = np.full(edge_ids.shape[0], np.nan, dtype=float)

    valid = pd.notna(edge_ids)
    if valid.any():
        valid_ids = edge_ids[valid].astype(int)
        valid = valid & (valid_ids >= 0)
        valid_ids = edge_ids[valid].astype(int)

        in_bounds = valid_ids < factor_matrix.shape[0]
        valid_positions = np.where(valid)[0][in_bounds]
        valid_ids = valid_ids[in_bounds]

        if mi_index >= factor_matrix.shape[1]:
            raise IndexError(
                f"mi_index={mi_index} exceeds factor_matrix.shape[1]={factor_matrix.shape[1]}"
            )

        out[valid_positions] = factor_matrix[valid_ids, mi_index]

    return out


def _gene_indices_from_names(var_names, genes):
    """Case-insensitive gene-name matching; preserve AnnData var_names capitalization."""
    var_names = pd.Index(var_names.astype(str))
    upper_to_idx = {}
    for i, g in enumerate(var_names):
        upper_to_idx.setdefault(str(g).upper(), i)

    idxs = []
    used_genes = []
    missing_genes = []

    seen = set()
    for g in genes:
        g = str(g).strip()
        if len(g) == 0:
            continue
        key = g.upper()
        if key in upper_to_idx:
            idx = upper_to_idx[key]
            if idx not in seen:
                idxs.append(idx)
                used_genes.append(str(var_names[idx]))
                seen.add(idx)
        else:
            missing_genes.append(g)

    return np.asarray(idxs, dtype=int), used_genes, missing_genes


# Cache for Scanpy AddModule-style scores.
# Keyed by expression object id and gene list so repeated positive/negative groups
# for the same sample/program do not recompute sc.tl.score_genes.
_ADD_MODULE_SCORE_CACHE = {}


# Cache for global z-score statistics.
# Keyed by reference cell type and gene list so repeated positive/negative groups
# reuse the same across-slice gene mean/std.
_GLOBAL_ZSCORE_STATS_CACHE = {}


# Across-slice reference statistics for gene-program scores.
def _global_zscore_stats_for_gene_names(
    gene_names,
    reference_cell_type=None,
):
    """Compute across-slice gene-wise mean/std for a reference cell population.

    This is used by MODULE_SCORE_METHOD='zscore_mean_global'.

    For example, for Cell2 CAF scoring:
      - reference_cell_type = ct2_cur, e.g. 'Fibroblast'
      - all Fibroblast cells from all slices are concatenated conceptually
      - one global mean/std is computed for each requested gene
      - each slice then uses these same global statistics for z-scoring

    The function accumulates sums / sums of squares / counts without explicitly
    concatenating all cells into one dense matrix.
    """
    method_gene_list = []
    seen = set()

    for g in gene_names:
        g = str(g).strip()
        if len(g) == 0:
            continue
        key = g.upper()
        if key not in seen:
            method_gene_list.append(g)
            seen.add(key)

    if len(method_gene_list) == 0:
        return (
            np.asarray([], dtype=float),
            np.asarray([], dtype=float),
            [],
            {
                "reference_cell_type": reference_cell_type,
                "n_reference_cells_total": 0,
                "n_genes_requested": 0,
                "n_genes_with_reference_values": 0,
            },
        )

    cache_key = (
        "zscore_mean_global",
        str(reference_cell_type) if reference_cell_type is not None else "__all_cells__",
        tuple(g.upper() for g in method_gene_list),
        int(len(globals().get("adata_list", []))),
    )

    if cache_key in _GLOBAL_ZSCORE_STATS_CACHE:
        return _GLOBAL_ZSCORE_STATS_CACHE[cache_key]

    required_globals = [
        "adata_list",
        "Factor_envir_list",
        "result",
        "SpiderNet_data_pyg_list",
        "cascade_def",
        "analysis_config",
        "_pathway_index_analyzer",
        "spna",
    ]

    missing_globals = [name for name in required_globals if name not in globals()]
    if len(missing_globals) > 0:
        raise NameError(
            "MODULE_SCORE_METHOD='zscore_mean_global' requires these objects "
            f"to be available before scoring: {missing_globals}"
        )

    n_genes = len(method_gene_list)
    gene_sum = np.zeros(n_genes, dtype=float)
    gene_sumsq = np.zeros(n_genes, dtype=float)
    gene_count = np.zeros(n_genes, dtype=float)

    n_reference_cells_total = 0
    sample_records = []

    for sample_index, (adata_cur, factor_arr_cur, hyper_adj_cur, sp_data_cur) in enumerate(
        zip(
            adata_list,
            Factor_envir_list,
            result["hyper_edge_adj_list"],
            SpiderNet_data_pyg_list,
        )
    ):
        sample_name_cur = _sample_name_from_adata(adata_cur, sample_index)

        sample_cur = spna.build_sample_prepared_data(
            adata=adata_cur,
            factor_envir_norm=factor_arr_cur,
            hyper_edge_adj=hyper_adj_cur,
            spidernet_data=sp_data_cur,
            cascade_definition=cascade_def,
            expression_max=_pathway_index_analyzer.expression_max,
            lr_max=_pathway_index_analyzer.lr_max,
            config=analysis_config,
        )

        cell_types_cur = np.asarray(sample_cur.cell_types).astype(str)

        if reference_cell_type is None:
            reference_nodes_cur = np.arange(sample_cur.expression.shape[0], dtype=int)
        else:
            reference_nodes_cur = np.where(cell_types_cur == str(reference_cell_type))[0]

        n_reference_cells_total += int(len(reference_nodes_cur))

        if len(reference_nodes_cur) == 0:
            sample_records.append({
                "sample_index": int(sample_index),
                "sample_name": str(sample_name_cur),
                "n_reference_cells": 0,
                "n_requested_genes_found": 0,
            })
            continue

        # Match requested genes to this sample's var_names in the same order
        # as method_gene_list. Missing genes in a sample are skipped for that sample.
        var_index_cur = pd.Index(adata_cur.var_names.astype(str))
        upper_to_idx_cur = {}
        for idx_cur, gene_cur in enumerate(var_index_cur):
            upper_to_idx_cur.setdefault(str(gene_cur).upper(), int(idx_cur))

        positions = []
        gene_indices_cur = []

        for pos, gene_name in enumerate(method_gene_list):
            key = str(gene_name).upper()
            if key in upper_to_idx_cur:
                positions.append(pos)
                gene_indices_cur.append(upper_to_idx_cur[key])

        if len(gene_indices_cur) == 0:
            sample_records.append({
                "sample_index": int(sample_index),
                "sample_name": str(sample_name_cur),
                "n_reference_cells": int(len(reference_nodes_cur)),
                "n_requested_genes_found": 0,
            })
            continue

        x_ref = _matrix_subset_to_numpy(
            sample_cur.expression,
            reference_nodes_cur,
            np.asarray(gene_indices_cur, dtype=int),
        )

        finite = np.isfinite(x_ref)
        x_ref_zero = np.where(finite, x_ref, 0.0)

        positions = np.asarray(positions, dtype=int)
        gene_sum[positions] += np.sum(x_ref_zero, axis=0)
        gene_sumsq[positions] += np.sum(x_ref_zero * x_ref_zero, axis=0)
        gene_count[positions] += np.sum(finite, axis=0)

        sample_records.append({
            "sample_index": int(sample_index),
            "sample_name": str(sample_name_cur),
            "n_reference_cells": int(len(reference_nodes_cur)),
            "n_requested_genes_found": int(len(gene_indices_cur)),
        })

    gene_mean = np.full(n_genes, np.nan, dtype=float)
    gene_std = np.full(n_genes, np.nan, dtype=float)

    valid_mean = gene_count > 0
    gene_mean[valid_mean] = gene_sum[valid_mean] / gene_count[valid_mean]

    valid_std = gene_count >= 2
    # Unbiased sample variance, matching np.nanstd(..., ddof=1).
    variance = np.full(n_genes, np.nan, dtype=float)
    variance[valid_std] = (
        gene_sumsq[valid_std]
        - (gene_sum[valid_std] * gene_sum[valid_std] / gene_count[valid_std])
    ) / (gene_count[valid_std] - 1)

    variance[(~np.isfinite(variance)) | (variance < 0)] = np.nan
    gene_std[valid_std] = np.sqrt(variance[valid_std])
    gene_std[(~np.isfinite(gene_std)) | (gene_std == 0)] = np.nan

    info = {
        "reference_cell_type": reference_cell_type,
        "n_reference_cells_total": int(n_reference_cells_total),
        "n_genes_requested": int(n_genes),
        "n_genes_with_reference_values": int(np.sum(gene_count > 0)),
        "n_genes_with_nonzero_std": int(np.sum(np.isfinite(gene_std))),
        "sample_reference_summary": pd.DataFrame(sample_records),
    }

    result_tuple = (gene_mean, gene_std, method_gene_list, info)
    _GLOBAL_ZSCORE_STATS_CACHE[cache_key] = result_tuple

    print(
        "[Global z-score] "
        f"reference={reference_cell_type if reference_cell_type is not None else 'all cells'}, "
        f"reference cells={info['n_reference_cells_total']:,}, "
        f"genes with nonzero std={info['n_genes_with_nonzero_std']}/{info['n_genes_requested']}"
    )

    return result_tuple



# Module-score methods and within-sample AddModule-style scoring.
def _normalize_module_score_method(method):
    method = str(method).strip().lower().replace("-", "_")
    aliases = {
        "mean_expression": "mean",
        "raw_mean": "mean",
        "zscore": "zscore_mean",
        "z_score_mean": "zscore_mean",
        "zscored_mean": "zscore_mean",
        "zscore_global": "zscore_mean_global",
        "global_zscore": "zscore_mean_global",
        "global_zscore_mean": "zscore_mean_global",
        "z_score_mean_global": "zscore_mean_global",
        "zscored_mean_global": "zscore_mean_global",
        "global_zscored_mean": "zscore_mean_global",
        "add_module": "addmodule",
        "addmodulescore": "addmodule",
        "addmodule_score": "addmodule",
        "scanpy": "addmodule",
        "scanpy_score_genes": "addmodule",
        "score_genes": "addmodule",
    }
    return aliases.get(method, method)


def _addmodule_scores_all_nodes(
    expression,
    var_names,
    gene_names,
    ctrl_size=50,
    random_state=0,
    use_raw=False,
):
    """Compute Scanpy AddModule-style scores for all nodes in one sample.

    The function constructs a temporary AnnData from the current sample-level
    expression matrix, so the AddModule scores are computed within each
    slice/sample separately.
    """
    method_gene_list = []
    var_index = pd.Index(var_names).astype(str)
    var_set = set(var_index)

    for g in gene_names:
        g = str(g)
        if g in var_set and g not in method_gene_list:
            method_gene_list.append(g)

    if len(method_gene_list) == 0:
        n_obs = int(expression.shape[0])
        return np.full(n_obs, np.nan, dtype=float), []

    cache_key = (
        "addmodule",
        id(expression),
        tuple(method_gene_list),
        int(ctrl_size),
        int(random_state),
        bool(use_raw),
        tuple(expression.shape),
    )
    if cache_key in _ADD_MODULE_SCORE_CACHE:
        return _ADD_MODULE_SCORE_CACHE[cache_key], method_gene_list

    try:
        import scanpy as sc
        from anndata import AnnData
    except ImportError as exc:
        raise ImportError(
            "MODULE_SCORE_METHOD='addmodule' requires scanpy and anndata. "
            "Please install/import scanpy, or set MODULE_SCORE_METHOD to "
            "'mean' or 'zscore_mean'."
        ) from exc

    # Keep sparse matrices sparse. Convert torch/pandas objects only when needed.
    if sparse.issparse(expression):
        X_tmp = expression.copy()
    elif hasattr(expression, "detach"):
        X_tmp = expression.detach().cpu().numpy()
    elif hasattr(expression, "cpu") and hasattr(expression, "numpy"):
        X_tmp = expression.cpu().numpy()
    else:
        X_tmp = np.asarray(expression)

    adata_tmp = AnnData(X=X_tmp)
    adata_tmp.var_names = var_index
    adata_tmp.obs_names = [f"cell_{i}" for i in range(adata_tmp.n_obs)]

    score_name = "__addmodule_score__"
    sc.tl.score_genes(
        adata_tmp,
        gene_list=method_gene_list,
        score_name=score_name,
        ctrl_size=int(ctrl_size),
        use_raw=bool(use_raw),
        random_state=int(random_state),
    )

    scores = pd.to_numeric(adata_tmp.obs[score_name], errors="coerce").to_numpy(dtype=float)
    _ADD_MODULE_SCORE_CACHE[cache_key] = scores

    return scores, method_gene_list


def _module_score_for_nodes(
    expression,
    node_ids,
    gene_indices,
    method="zscore_mean",
    reference_nodes=None,
    var_names=None,
    gene_names=None,
    addmodule_ctrl_size=50,
    addmodule_random_state=0,
    addmodule_use_raw=False,
    global_reference_cell_type=None,
):
    """Compute module scores for selected nodes.

    Supported methods
    -----------------
    mean
        Direct average expression across genes.

    zscore_mean
        Gene-wise z-score within a sample-level reference cell population,
        then average genes.

    zscore_mean_global
        Gene-wise z-score using global mean/std computed from the matching
        reference cell population concatenated across all slices, then
        average genes for the selected nodes in the current slice.

    addmodule
        Scanpy AddModule-style score using sc.tl.score_genes. The score is
        computed on the current sample/slice, then extracted for node_ids.
    """
    method_norm = _normalize_module_score_method(method)

    node_ids = np.asarray(node_ids, dtype=int)
    gene_indices = np.asarray(gene_indices, dtype=int)

    if node_ids.size == 0:
        return np.asarray([], dtype=float)
    if gene_indices.size == 0:
        return np.full(node_ids.size, np.nan, dtype=float)

    if method_norm == "addmodule":
        if var_names is None:
            raise ValueError("var_names must be provided when MODULE_SCORE_METHOD='addmodule'.")

        if gene_names is None:
            var_index = pd.Index(var_names).astype(str)
            gene_names = [str(var_index[i]) for i in gene_indices if 0 <= int(i) < len(var_index)]

        all_scores, used_gene_names = _addmodule_scores_all_nodes(
            expression=expression,
            var_names=var_names,
            gene_names=gene_names,
            ctrl_size=addmodule_ctrl_size,
            random_state=addmodule_random_state,
            use_raw=addmodule_use_raw,
        )

        valid = (node_ids >= 0) & (node_ids < len(all_scores))
        out = np.full(node_ids.size, np.nan, dtype=float)
        out[valid] = all_scores[node_ids[valid]]
        return out

    if method_norm == "zscore_mean_global":
        if var_names is None:
            raise ValueError("var_names must be provided when MODULE_SCORE_METHOD='zscore_mean_global'.")

        var_index = pd.Index(var_names).astype(str)

        if gene_names is None or len(gene_names) != len(gene_indices):
            gene_names_for_stats = [
                str(var_index[int(i)])
                for i in gene_indices
                if 0 <= int(i) < len(var_index)
            ]
        else:
            gene_names_for_stats = [str(g) for g in gene_names]

        if len(gene_names_for_stats) == 0:
            return np.full(node_ids.size, np.nan, dtype=float)

        # Keep target expression columns aligned to gene_names_for_stats.
        # In the normal code path, gene_indices and gene_names_for_stats have
        # the same length/order because both come from _gene_indices_from_names().
        if len(gene_names_for_stats) != len(gene_indices):
            gene_indices_for_target, gene_names_for_stats, _ = _gene_indices_from_names(
                var_index,
                gene_names_for_stats,
            )
        else:
            gene_indices_for_target = gene_indices

        target_x = _matrix_subset_to_numpy(expression, node_ids, gene_indices_for_target)

        gene_mean, gene_std, global_gene_names, global_info = _global_zscore_stats_for_gene_names(
            gene_names=gene_names_for_stats,
            reference_cell_type=global_reference_cell_type,
        )

        if len(gene_mean) != target_x.shape[1]:
            raise ValueError(
                "Global z-score statistics and target expression matrix are not aligned. "
                f"len(gene_mean)={len(gene_mean)}, target_x.shape[1]={target_x.shape[1]}"
            )

        z = (target_x - gene_mean) / gene_std
        z[~np.isfinite(z)] = 0.0

        return np.nanmean(z, axis=1)

    target_x = _matrix_subset_to_numpy(expression, node_ids, gene_indices)

    if method_norm == "mean":
        return np.nanmean(target_x, axis=1)

    if method_norm != "zscore_mean":
        raise ValueError(
            "MODULE_SCORE_METHOD must be one of: 'mean', 'zscore_mean', "
            "'zscore_mean_global', or 'addmodule'."
        )

    if reference_nodes is None or len(reference_nodes) == 0:
        reference_nodes = np.arange(expression.shape[0], dtype=int)
    reference_nodes = np.asarray(reference_nodes, dtype=int)

    ref_x = _matrix_subset_to_numpy(expression, reference_nodes, gene_indices)

    gene_mean = np.nanmean(ref_x, axis=0)
    if ref_x.shape[0] >= 2:
        gene_std = np.nanstd(ref_x, axis=0, ddof=1)
    else:
        gene_std = np.full(ref_x.shape[1], np.nan)

    gene_std[(~np.isfinite(gene_std)) | (gene_std == 0)] = np.nan
    z = (target_x - gene_mean) / gene_std
    z[~np.isfinite(z)] = 0.0

    return np.nanmean(z, axis=1)


# Two-group tests and plot annotation conventions.
def _p_to_star(p):
    if not np.isfinite(p):
        return "n.s."
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "n.s."


def _compare_two_groups(x, y, paired=False, alternative="two-sided"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if paired:
        if len(x) != len(y) or len(x) < 2:
            return np.nan
        if not np.any(np.abs(x - y) > 0):
            return np.nan
        try:
            return float(wilcoxon(x, y, alternative=alternative, zero_method="wilcox").pvalue)
        except Exception:
            return np.nan

    if len(x) < 1 or len(y) < 1:
        return np.nan
    try:
        return float(mannwhitneyu(x, y, alternative=alternative).pvalue)
    except Exception:
        return np.nan


def _set_nature_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 8,
        "axes.linewidth": 0.8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def _wrap_label(x, width=26):
    return "\n".join(textwrap.wrap(str(x), width=width, break_long_words=False))


# Alternative figure implementation with transparent boxes (inactive).
# def plot_feature_panels_with_samplesize(
#     df,
#     value_col="value",
#     feature_col="feature",
#     group_col="group",
#     category_col="feature_category",
#     group_order=UPSTREAM_GATED_GROUPS,
#     pretty_group_labels=PRETTY_GROUP_LABELS_UPSTREAM_GATED,
#     colors=None,
#     ylabel="Score",
#     figure_title=None,
#     output_path=None,
#     pvalue_alternative="two-sided",
#     ncol=3,
#     figsize_per_panel=(1.4, 2.7),
#     title_wrap_width=22,
#     show=True,
#     log2fc_pseudocount=1e-9,
#     violin_alpha=0.80,
#     violin_width=0.80,
#     box_width=0.35,
#     boxplot_line_color="#bdbec1",
# ):
#     """Facet violin + transparent boxplots with n labeled on each box.
# 
#     Annotation:
#         stars (SMD=...)
# 
#     Styling:
#         - white figure and panel background
#         - violin plot behind boxplot
#         - no scatter points
#         - no p-value text
#         - no log2FC text
#         - no boxplot caps
#         - boxplot fill is transparent / no color
#         - boxplot rectangle border, whiskers, and median line use dark gray
#     """
#     import os
#     import math
#     import numpy as np
#     import pandas as pd
#     import matplotlib.pyplot as plt
# 
#     _set_nature_style()
# 
#     plt.rcParams.update({
#         "figure.facecolor": "white",
#         "axes.facecolor": "white",
#         "savefig.facecolor": "white",
#         "axes.edgecolor": "black",
#         "axes.grid": False,
#         "grid.alpha": 0.0,
#     })
# 
#     if len(group_order) != 2:
#         raise ValueError("plot_feature_panels_with_samplesize currently expects exactly two groups.")
# 
#     # ----------------------------------------------------------
#     # Colors
#     # ----------------------------------------------------------
#     # Violin fill colors.
#     if colors is None:
#         colors = {
#             group_order[0]: "#aad58e",
#             group_order[1]: "#f1f5d3",
#         }
# 
#     # Violin outline colors. Keep separate from boxplot line color.
#     violin_edge_colors = {
#         group_order[0]: "#5f8f4e",
#         group_order[1]: "#d1cabd",
#     }
# 
#     # ----------------------------------------------------------
#     # Local helper: Tukey whisker positions
#     # ----------------------------------------------------------
#     def _boxplot_whiskers(arr):
#         """Return lower and upper whisker using Tukey 1.5 IQR rule."""
#         arr = np.asarray(arr, dtype=float)
#         arr = arr[np.isfinite(arr)]
# 
#         if len(arr) == 0:
#             return np.nan, np.nan
# 
#         q1 = np.percentile(arr, 25)
#         q3 = np.percentile(arr, 75)
#         iqr = q3 - q1
# 
#         lower_bound = q1 - 1.5 * iqr
#         upper_bound = q3 + 1.5 * iqr
# 
#         lower_candidates = arr[arr >= lower_bound]
#         upper_candidates = arr[arr <= upper_bound]
# 
#         lower_whisker = np.min(lower_candidates) if len(lower_candidates) > 0 else np.min(arr)
#         upper_whisker = np.max(upper_candidates) if len(upper_candidates) > 0 else np.max(arr)
# 
#         return float(lower_whisker), float(upper_whisker)
# 
#     # ----------------------------------------------------------
#     # Local helper: standardized mean difference
#     # ----------------------------------------------------------
#     def _standardized_mean_difference(x, y):
#         """
#         SMD = (mean(x) - mean(y)) / pooled SD
#         """
#         x = np.asarray(x, dtype=float)
#         y = np.asarray(y, dtype=float)
# 
#         x = x[np.isfinite(x)]
#         y = y[np.isfinite(y)]
# 
#         n1 = len(x)
#         n2 = len(y)
# 
#         if n1 < 2 or n2 < 2:
#             return np.nan
# 
#         mean_x = float(np.nanmean(x))
#         mean_y = float(np.nanmean(y))
# 
#         sd_x = float(np.nanstd(x, ddof=1))
#         sd_y = float(np.nanstd(y, ddof=1))
# 
#         pooled_var = ((n1 - 1) * sd_x**2 + (n2 - 1) * sd_y**2) / (n1 + n2 - 2)
# 
#         if not np.isfinite(pooled_var) or pooled_var <= 0:
#             return np.nan
# 
#         pooled_sd = np.sqrt(pooled_var)
#         return float((mean_x - mean_y) / pooled_sd)
# 
#     # ----------------------------------------------------------
#     # Prepare data
#     # ----------------------------------------------------------
#     plot_df = df.copy()
#     plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors="coerce")
#     plot_df = plot_df[
#         plot_df[group_col].isin(group_order) &
#         plot_df[value_col].notna()
#     ].copy()
# 
#     if plot_df.empty:
#         raise ValueError("No values available for plotting.")
# 
#     if category_col in plot_df.columns:
#         plot_df["_facet"] = (
#             plot_df[category_col].astype(str)
#             + " | "
#             + plot_df[feature_col].astype(str)
#         )
#     else:
#         plot_df["_facet"] = plot_df[feature_col].astype(str)
# 
#     facets = list(pd.unique(plot_df["_facet"]))
# 
#     n_facets = len(facets)
#     ncol = 3
#     nrow = int(math.ceil(n_facets / ncol))
# 
#     fig_w = max(3.2, figsize_per_panel[0] * ncol)
#     fig_h = max(3.0, figsize_per_panel[1] * nrow)
# 
#     fig, axes = plt.subplots(
#         nrow,
#         ncol,
#         figsize=(fig_w, fig_h),
#         squeeze=False,
#         sharey=False,
#         facecolor="white",
#     )
#     fig.patch.set_facecolor("white")
# 
#     # ----------------------------------------------------------
#     # Plot each panel
#     # ----------------------------------------------------------
#     for i, facet in enumerate(facets):
#         ax = axes.flat[i]
#         ax.set_facecolor("white")
# 
#         sub = plot_df[plot_df["_facet"].eq(facet)].copy()
# 
#         data = []
#         ns = []
# 
#         for g in group_order:
#             arr = sub.loc[sub[group_col].eq(g), value_col].to_numpy(dtype=float)
#             arr = arr[np.isfinite(arr)]
#             data.append(arr)
#             ns.append(len(arr))
# 
#         # ------------------------------------------------------
#         # Violin plot, behind boxplot
#         # ------------------------------------------------------
#         for xpos, (arr, g) in enumerate(zip(data, group_order), start=1):
#             if len(arr) < 2:
#                 continue
# 
#             vp = ax.violinplot(
#                 dataset=[arr],
#                 positions=[xpos],
#                 widths=violin_width,
#                 showmeans=False,
#                 showmedians=False,
#                 showextrema=False,
#             )
# 
#             for body in vp["bodies"]:
#                 body.set_facecolor(colors.get(g, "white"))
#                 body.set_edgecolor(violin_edge_colors.get(g, boxplot_line_color))
#                 body.set_linewidth(0.8)
#                 body.set_alpha(violin_alpha)
#                 body.set_zorder(1)
# 
#         # ------------------------------------------------------
#         # Boxplot overlay: transparent fill, dark-gray lines
#         # ------------------------------------------------------
#         bp = ax.boxplot(
#             data,
#             patch_artist=True,
#             showfliers=False,
#             showcaps=False,
#             widths=box_width,
#             medianprops=dict(color=boxplot_line_color, linewidth=1.0),
#             boxprops=dict(edgecolor=boxplot_line_color, linewidth=0.9, facecolor="none"),
#             whiskerprops=dict(color=boxplot_line_color, linewidth=0.8),
#             zorder=3,
#         )
# 
#         # Box fill transparent; border dark gray.
#         for box in bp["boxes"]:
#             box.set_facecolor("none")
#             box.set_alpha(1.0)
#             box.set_edgecolor(boxplot_line_color)
#             box.set_linewidth(0.9)
#             box.set_zorder(3)
# 
#         # Median line: dark gray.
#         for median_line in bp["medians"]:
#             median_line.set_color(boxplot_line_color)
#             median_line.set_linewidth(1.0)
#             median_line.set_zorder(4)
# 
#         # Whisker vertical lines: dark gray.
#         for whisker in bp["whiskers"]:
#             whisker.set_color(boxplot_line_color)
#             whisker.set_linewidth(0.8)
#             whisker.set_zorder(3)
# 
#         # ------------------------------------------------------
#         # P-value stars
#         # ------------------------------------------------------
#         pval = _compare_two_groups(
#             data[0],
#             data[1],
#             paired=False,
#             alternative=pvalue_alternative,
#         )
# 
#         star = _p_to_star(pval) if np.isfinite(pval) else "n.s."
# 
#         # ------------------------------------------------------
#         # Standardized mean difference
#         # ------------------------------------------------------
#         smd = _standardized_mean_difference(data[0], data[1])
#         smd_text = f"SMD={smd:+.2f}" if np.isfinite(smd) else "SMD=NA"
# 
#         text = f"{star} ({smd_text})"
# 
#         # ------------------------------------------------------
#         # Whisker-based ylim and annotation height
#         # ------------------------------------------------------
#         lower_whiskers = []
#         upper_whiskers = []
# 
#         for arr in data:
#             lw, uw = _boxplot_whiskers(arr)
#             if np.isfinite(lw):
#                 lower_whiskers.append(lw)
#             if np.isfinite(uw):
#                 upper_whiskers.append(uw)
# 
#         if len(lower_whiskers) > 0 and len(upper_whiskers) > 0:
#             lower_whisker_min = float(np.min(lower_whiskers))
#             upper_whisker_max = float(np.max(upper_whiskers))
#         else:
#             finite_arrays = [arr for arr in data if len(arr) > 0]
#             if finite_arrays:
#                 allv = np.concatenate(finite_arrays)
#                 lower_whisker_min = float(np.nanmin(allv))
#                 upper_whisker_max = float(np.nanmax(allv))
#             else:
#                 lower_whisker_min = 0.0
#                 upper_whisker_max = 1.0
# 
#         d = upper_whisker_max - lower_whisker_min
#         if d == 0:
#             d = max(abs(upper_whisker_max), 1.0) * 0.1
# 
#         vmin = lower_whisker_min - 0.10 * d
#         vmax = upper_whisker_max + 0.10 * d
#         annotation_y = upper_whisker_max + 0.05 * d
#         bracket_h = 0.015 * d
# 
#         ax.plot(
#             [1, 1, 2, 2],
#             [
#                 annotation_y - bracket_h,
#                 annotation_y,
#                 annotation_y,
#                 annotation_y - bracket_h,
#             ],
#             color="black",
#             lw=0.7,
#             clip_on=False,
#             zorder=5,
#         )
# 
#         ax.text(
#             1.5,
#             annotation_y + 0.008 * d,
#             text,
#             ha="center",
#             va="bottom",
#             fontsize=7,
#             clip_on=False,
#             zorder=6,
#         )
# 
#         ax.set_ylim(vmin, vmax)
# 
#         # ------------------------------------------------------
#         # Axis labels
#         # ------------------------------------------------------
#         xticklabels = [
#             f"{pretty_group_labels.get(g, g)}\nn={n}"
#             for g, n in zip(group_order, ns)
#         ]
# 
#         ax.set_xticks([1, 2])
#         ax.set_xticklabels(
#             xticklabels,
#             rotation=20,
#             ha="right",
#         )
# 
#         ax.set_ylabel(ylabel)
#         ax.set_title(
#             _wrap_label(facet, width=title_wrap_width),
#             pad=6,
#             fontsize=7,
#         )
# 
#         # ------------------------------------------------------
#         # Explicit x/y axes; no gray background
#         # ------------------------------------------------------
#         ax.grid(False)
# 
#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
# 
#         ax.spines["left"].set_visible(True)
#         ax.spines["bottom"].set_visible(True)
#         ax.spines["left"].set_color("black")
#         ax.spines["bottom"].set_color("black")
#         ax.spines["left"].set_linewidth(0.8)
#         ax.spines["bottom"].set_linewidth(0.8)
# 
#         ax.tick_params(
#             axis="both",
#             which="both",
#             direction="out",
#             length=3,
#             width=0.8,
#             color="black",
#             labelcolor="black",
#         )
# 
#     # Empty panels off and white.
#     for j in range(n_facets, nrow * ncol):
#         axes.flat[j].set_facecolor("white")
#         axes.flat[j].axis("off")
# 
#     if figure_title:
#         fig.suptitle(figure_title, y=1.01, fontsize=10)
# 
#     fig.tight_layout()
# 
#     # ----------------------------------------------------------
#     # Save
#     # ----------------------------------------------------------
#     if output_path is not None:
#         output_path = str(output_path)
#         root, ext = os.path.splitext(output_path)
# 
#         if ext.lower() in (".pdf", ".png", ".svg", ".jpg", ".jpeg"):
#             fig.savefig(
#                 output_path,
#                 bbox_inches="tight",
#                 facecolor="white",
#                 edgecolor="white",
#             )
#             print(f"Saved: {output_path}")
#         else:
#             fig.savefig(
#                 f"{output_path}.pdf",
#                 bbox_inches="tight",
#                 facecolor="white",
#                 edgecolor="white",
#             )
#             fig.savefig(
#                 f"{output_path}.png",
#                 dpi=300,
#                 bbox_inches="tight",
#                 facecolor="white",
#                 edgecolor="white",
#             )
#             print(f"Saved: {output_path}.pdf")
#             print(f"Saved: {output_path}.png")
# 
#     if show:
#         plt.show()
# 
#     plt.close(fig)
#     
    
def plot_feature_panels_with_samplesize(
    df,
    value_col="value",
    feature_col="feature",
    group_col="group",
    category_col="feature_category",
    group_order=UPSTREAM_GATED_GROUPS,
    pretty_group_labels=PRETTY_GROUP_LABELS_UPSTREAM_GATED,
    colors=None,
    ylabel="Score",
    figure_title=None,
    output_path=None,
    pvalue_alternative="two-sided",
    ncol=3,
    figsize_per_panel=(1.4, 2.7),
    title_wrap_width=22,
    show=True,
    log2fc_pseudocount=1e-9,
    violin_alpha=0.50,
    violin_width=0.85,
    box_width=0.35,
    boxplot_line_color="#bdbec1",
):
    """Plot two-group feature distributions with sample sizes and effect sizes.

    The layout uses three columns, overriding ncol. Each panel shows a violin
    and filled boxplot, with unpaired Mann-Whitney U significance stars and
    SMD = (mean of group 0 - mean of group 1) / pooled sample SD. Raw P-values
    and the computed log2 mean ratio are not displayed. The same unpaired
    annotation is used for sample-level input, regardless of the test used
    in the separately exported statistics table.

    Axis limits extend 20% beyond the range of Tukey whisker endpoints;
    annotations sit 15% above the upper endpoint. Outliers and caps are hidden.
    """
    import os
    import math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    _set_nature_style()

    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.edgecolor": "black",
        "axes.grid": False,
        "grid.alpha": 0.0,
    })

    if len(group_order) != 2:
        raise ValueError("plot_feature_panels_with_samplesize currently expects exactly two groups.")

    if colors is None:
        colors = {
            group_order[0]: "#aad58e",
            group_order[1]: "#f1f5d3",
        }

    edge_colors = {
        group_order[0]: "#aad58e",
        group_order[1]: "#f1f5d3",
    }

    def _boxplot_whiskers(arr):
        """Return lower and upper whisker using Tukey 1.5 IQR rule."""
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]

        if len(arr) == 0:
            return np.nan, np.nan

        q1 = np.percentile(arr, 25)
        q3 = np.percentile(arr, 75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        lower_candidates = arr[arr >= lower_bound]
        upper_candidates = arr[arr <= upper_bound]

        lower_whisker = np.min(lower_candidates) if len(lower_candidates) > 0 else np.min(arr)
        upper_whisker = np.max(upper_candidates) if len(upper_candidates) > 0 else np.max(arr)

        return float(lower_whisker), float(upper_whisker)

    def _standardized_mean_difference(x, y):
        """
        SMD = (mean(x) - mean(y)) / pooled SD
        """
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        n1 = len(x)
        n2 = len(y)

        if n1 < 2 or n2 < 2:
            return np.nan

        mean_x = float(np.nanmean(x))
        mean_y = float(np.nanmean(y))

        sd_x = float(np.nanstd(x, ddof=1))
        sd_y = float(np.nanstd(y, ddof=1))

        pooled_var = ((n1 - 1) * sd_x**2 + (n2 - 1) * sd_y**2) / (n1 + n2 - 2)

        if not np.isfinite(pooled_var) or pooled_var <= 0:
            return np.nan

        pooled_sd = np.sqrt(pooled_var)
        return float((mean_x - mean_y) / pooled_sd)

    plot_df = df.copy()
    plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors="coerce")
    plot_df = plot_df[
        plot_df[group_col].isin(group_order) &
        plot_df[value_col].notna()
    ].copy()

    if plot_df.empty:
        raise ValueError("No values available for plotting.")

    if category_col in plot_df.columns:
        plot_df["_facet"] = (
            plot_df[category_col].astype(str)
            + " | "
            + plot_df[feature_col].astype(str)
        )
    else:
        plot_df["_facet"] = plot_df[feature_col].astype(str)

    facets = list(pd.unique(plot_df["_facet"]))

    n_facets = len(facets)
    ncol = 3
    nrow = int(math.ceil(n_facets / ncol))

    fig_w = max(3.2, figsize_per_panel[0] * ncol)
    fig_h = max(3.0, figsize_per_panel[1] * nrow)

    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(fig_w, fig_h),
        squeeze=False,
        sharey=False,
        facecolor="white",
    )
    fig.patch.set_facecolor("white")

    for i, facet in enumerate(facets):
        ax = axes.flat[i]
        ax.set_facecolor("white")

        sub = plot_df[plot_df["_facet"].eq(facet)].copy()

        data = []
        ns = []

        for g in group_order:
            arr = sub.loc[sub[group_col].eq(g), value_col].to_numpy(dtype=float)
            arr = arr[np.isfinite(arr)]
            data.append(arr)
            ns.append(len(arr))

        # ------------------------------------------------------
        # Violin plot, behind boxplot
        # ------------------------------------------------------
        for xpos, (arr, g) in enumerate(zip(data, group_order), start=1):
            if len(arr) < 2:
                continue

            vp = ax.violinplot(
                dataset=[arr],
                positions=[xpos],
                widths=violin_width,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )

            for body in vp["bodies"]:
                body.set_facecolor(colors.get(g, "white"))
                body.set_edgecolor(edge_colors.get(g, "black"))
                body.set_linewidth(0.8)
                body.set_alpha(violin_alpha)
                body.set_zorder(1)

        # ------------------------------------------------------
        # Boxplot overlay
        # ------------------------------------------------------
        bp = ax.boxplot(
            data,
            patch_artist=True,
            showfliers=False,
            showcaps=False,
            widths=box_width,
            medianprops=dict(color=boxplot_line_color, linewidth=0.9),
            boxprops=dict(edgecolor=boxplot_line_color, linewidth=0.9),
            whiskerprops=dict(color=boxplot_line_color, linewidth=0.8),
            zorder=3,
        )

        for box, g in zip(bp["boxes"], group_order):
            box.set_facecolor(colors.get(g, "white"))
            box.set_edgecolor(boxplot_line_color)
            box.set_linewidth(0.9)
            box.set_zorder(3)

        for median_line in bp["medians"]:
            median_line.set_color(boxplot_line_color)
            median_line.set_linewidth(0.9)
            median_line.set_zorder(4)

        for whisker in bp["whiskers"]:
            whisker.set_color(boxplot_line_color)
            whisker.set_linewidth(0.8)
            whisker.set_zorder(3)

        # ------------------------------------------------------
        # P-value stars
        # ------------------------------------------------------
        pval = _compare_two_groups(
            data[0],
            data[1],
            paired=False,
            alternative=pvalue_alternative,
        )

        star = _p_to_star(pval) if np.isfinite(pval) else "n.s."

        # ------------------------------------------------------
        # log2 fold change of mean
        # ------------------------------------------------------
        mean_1 = float(np.nanmean(data[0])) if len(data[0]) > 0 else np.nan
        mean_2 = float(np.nanmean(data[1])) if len(data[1]) > 0 else np.nan

        log2fc_mean = np.nan
        if np.isfinite(mean_1) and np.isfinite(mean_2):
            numerator = mean_1 + log2fc_pseudocount
            denominator = mean_2 + log2fc_pseudocount

            if numerator > 0 and denominator > 0:
                log2fc_mean = float(np.log2(numerator / denominator))

        fc_text = f"{log2fc_mean:+.2f}" if np.isfinite(log2fc_mean) else "NA"

        # ------------------------------------------------------
        # SMD
        # ------------------------------------------------------
        smd = _standardized_mean_difference(data[0], data[1])
        smd_text = f"SMD={smd:+.2f}" if np.isfinite(smd) else "SMD=NA"

        # text = f"{star} ({fc_text}) [{smd_text}]"
        text = f"{star} ({smd_text})"

        # ------------------------------------------------------
        # Whisker-based ylim and annotation height
        # ------------------------------------------------------
        lower_whiskers = []
        upper_whiskers = []

        for arr in data:
            lw, uw = _boxplot_whiskers(arr)
            if np.isfinite(lw):
                lower_whiskers.append(lw)
            if np.isfinite(uw):
                upper_whiskers.append(uw)

        if len(lower_whiskers) > 0 and len(upper_whiskers) > 0:
            lower_whisker_min = float(np.min(lower_whiskers))
            upper_whisker_max = float(np.max(upper_whiskers))
        else:
            finite_arrays = [arr for arr in data if len(arr) > 0]
            if finite_arrays:
                allv = np.concatenate(finite_arrays)
                lower_whisker_min = float(np.nanmin(allv))
                upper_whisker_max = float(np.nanmax(allv))
            else:
                lower_whisker_min = 0.0
                upper_whisker_max = 1.0

        d = upper_whisker_max - lower_whisker_min
        if d == 0:
            d = max(abs(upper_whisker_max), 1.0) * 0.1

        vmin = lower_whisker_min - 0.20 * d
        vmax = upper_whisker_max + 0.20 * d
        annotation_y = upper_whisker_max + 0.15 * d
        bracket_h = 0.015 * d

        ax.plot(
            [1, 1, 2, 2],
            [
                annotation_y - bracket_h,
                annotation_y,
                annotation_y,
                annotation_y - bracket_h,
            ],
            color="black",
            lw=0.7,
            clip_on=False,
            zorder=5,
        )

        ax.text(
            1.5,
            annotation_y + 0.008 * d,
            text,
            ha="center",
            va="bottom",
            fontsize=7,
            clip_on=False,
            zorder=6,
        )

        ax.set_ylim(vmin, vmax)

        xticklabels = [
            f"{pretty_group_labels.get(g, g)}\nn={n}"
            for g, n in zip(group_order, ns)
        ]

        ax.set_xticks([1, 2])
        ax.set_xticklabels(
            xticklabels,
            rotation=20,
            ha="right",
        )

        ax.set_ylabel(ylabel)
        ax.set_title(
            _wrap_label(facet, width=title_wrap_width),
            pad=6,
            fontsize=7,
        )

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_visible(True)
        ax.spines["left"].set_color("black")
        ax.spines["bottom"].set_color("black")
        ax.spines["left"].set_linewidth(0.8)
        ax.spines["bottom"].set_linewidth(0.8)

        ax.tick_params(
            axis="both",
            which="both",
            direction="out",
            length=3,
            width=0.8,
            color="black",
            labelcolor="black",
        )

    for j in range(n_facets, nrow * ncol):
        axes.flat[j].set_facecolor("white")
        axes.flat[j].axis("off")

    if figure_title:
        fig.suptitle(figure_title, y=1.01, fontsize=10)

    fig.tight_layout()

    if output_path is not None:
        output_path = str(output_path)
        root, ext = os.path.splitext(output_path)

        if ext.lower() in (".pdf", ".png", ".svg", ".jpg", ".jpeg"):
            fig.savefig(
                output_path,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
            )
            print(f"Saved: {output_path}")
        else:
            fig.savefig(
                f"{output_path}.pdf",
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
            )
            fig.savefig(
                f"{output_path}.png",
                dpi=300,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
            )
            print(f"Saved: {output_path}.pdf")
            print(f"Saved: {output_path}.png")

    if show:
        plt.show()

    plt.close(fig)
# --------------------------------------------------------------
# Directed edge mapping and triplet validation
# --------------------------------------------------------------
def _build_directed_edge_lookup_from_edge_index(edge_index):
    """Build {(source_cell, target_cell): edge_id} from sample.edge_index.

    The edge_id is the row index in edge_index after conversion to n_edges×2,
    therefore it is aligned with factor_matrix and lr_matrix rows.
    If duplicate directed pairs exist, the first edge id is kept and duplicates
    are reported.  Duplicate directed pairs should be rare; if frequent, inspect
    the processed graph.
    """
    edges = _edge_array_from_edge_index(edge_index)
    lookup = {}
    duplicate_pairs = {}

    for edge_id, (src, dst) in enumerate(edges[:, :2]):
        key = (int(src), int(dst))
        if key in lookup:
            duplicate_pairs[key] = duplicate_pairs.get(key, 1) + 1
            continue
        lookup[key] = int(edge_id)

    return lookup, duplicate_pairs


def _triplets_to_directed_edge_ids(
    triplets,
    edge_index,
    edge="upstream",
    directed_edge_lookup=None,
):
    """Map each triplet to exactly one directed spatial edge id.

    Missing mappings are returned as np.nan instead of being dropped.  This
    preserves triplet alignment and lets the caller explicitly validate/drop
    invalid triplets.

    edge="upstream":   cell1 -> cell2
    edge="downstream": cell2 -> cell3
    """
    triplets = np.asarray(triplets, dtype=int)
    if triplets.ndim != 2 or triplets.shape[1] < 3:
        raise ValueError(f"triplets must have shape n_triplets×>=3; got {triplets.shape}")

    if directed_edge_lookup is None:
        directed_edge_lookup, _ = _build_directed_edge_lookup_from_edge_index(edge_index)

    edge_ids = np.full(triplets.shape[0], np.nan, dtype=float)

    for i, trip in enumerate(triplets):
        c1, c2, c3 = [int(x) for x in trip[:3]]

        if edge == "upstream":
            key = (c1, c2)
        elif edge == "downstream":
            key = (c2, c3)
        else:
            raise ValueError("edge must be either 'upstream' or 'downstream'.")

        edge_id = directed_edge_lookup.get(key, None)
        if edge_id is not None:
            edge_ids[i] = int(edge_id)

    return edge_ids


def _validate_and_filter_triplet_edges(
    triplets,
    edge12_ids,
    edge23_ids,
    sample_index,
    sample_name,
    group,
    max_drop_fraction=0.05,
    raise_on_high_dropout=True,
):
    """Keep only triplets with both directed spatial edges resolved.

    Retained triplets must satisfy both directed neighbor relations,
    cell1->cell2 and cell2->cell3. Unresolved triplets are dropped; exceeding
    max_drop_fraction raises an error when raise_on_high_dropout is enabled.
    """
    triplets = np.asarray(triplets, dtype=int)
    edge12_ids = np.asarray(edge12_ids, dtype=float)
    edge23_ids = np.asarray(edge23_ids, dtype=float)

    n_triplets_before = int(triplets.shape[0])

    if edge12_ids.shape[0] != n_triplets_before or edge23_ids.shape[0] != n_triplets_before:
        raise ValueError(
            "Edge-id arrays must be aligned one-to-one with triplets before validation. "
            f"n_triplets={n_triplets_before}, len(edge12_ids)={edge12_ids.shape[0]}, "
            f"len(edge23_ids)={edge23_ids.shape[0]}"
        )

    valid12 = np.isfinite(edge12_ids)
    valid23 = np.isfinite(edge23_ids)
    valid_mask = valid12 & valid23

    n_missing_edge12 = int(np.sum(~valid12))
    n_missing_edge23 = int(np.sum(~valid23))
    n_missing_either = int(np.sum(~valid_mask))
    n_triplets_after = int(np.sum(valid_mask))
    drop_fraction = float(n_missing_either / max(n_triplets_before, 1))

    validation_record = {
        "sample_index": int(sample_index),
        "sample_name": str(sample_name),
        "group": str(group),
        "n_triplets_before_edge_validation": n_triplets_before,
        "n_missing_edge12": n_missing_edge12,
        "n_missing_edge23": n_missing_edge23,
        "n_missing_either_edge": n_missing_either,
        "n_triplets_after_edge_validation": n_triplets_after,
        "edge_mapping_drop_fraction": drop_fraction,
        "max_allowed_drop_fraction": float(max_drop_fraction),
    }

    if n_missing_either > 0:
        msg = (
            f"[Edge validation] sample={sample_name}, group={group}: "
            f"dropping {n_missing_either}/{n_triplets_before} triplets "
            f"({drop_fraction:.2%}) because edge12 or edge23 could not be "
            "mapped to a directed spatial edge."
        )
        print(msg)

    if (
        n_triplets_before > 0
        and drop_fraction > max_drop_fraction
        and raise_on_high_dropout
    ):
        raise ValueError(
            f"Too many triplets failed directed-edge validation for "
            f"sample={sample_name}, group={group}: "
            f"{n_missing_either}/{n_triplets_before} = {drop_fraction:.2%}. "
            "This is likely not a small artifact. Check edge direction, "
            "sample.edge_index, spna.extract_triplets(), and the cell-type/C5 filters."
        )

    return (
        triplets[valid_mask],
        edge12_ids[valid_mask].astype(int),
        edge23_ids[valid_mask].astype(int),
        validation_record,
    )



# --------------------------------------------------------------
# Cross-group overlap exclusion
# --------------------------------------------------------------
def _normalise_overlap_edge_roles(edge_roles):
    if edge_roles is None:
        return ("upstream", "downstream")
    edge_roles = tuple(str(x) for x in edge_roles)
    valid = {"upstream", "downstream"}
    unknown = sorted(set(edge_roles) - valid)
    if unknown:
        raise ValueError(f"Unknown edge roles for overlap exclusion: {unknown}. Expected subset of {valid}.")
    return edge_roles


def _normalise_overlap_cell_positions(cell_positions):
    if cell_positions is None:
        return (0, 1, 2)
    cell_positions = tuple(int(x) for x in cell_positions)
    valid = {0, 1, 2}
    unknown = sorted(set(cell_positions) - valid)
    if unknown:
        raise ValueError(f"Unknown cell positions for overlap exclusion: {unknown}. Expected subset of {valid}.")
    return cell_positions


def _triplet_unit_sets_for_overlap(
    triplets,
    edge12_ids,
    edge23_ids,
    edge_roles=("upstream", "downstream"),
    cell_positions=(0, 1, 2),
):
    """Collect edge and cell units used by a group of triplets.

    Edge ids are the row ids in sample.edge_index.  Cell ids are local AnnData
    row indices in the current sample.  The output sets are therefore comparable
    only within the same sample.
    """
    triplets = np.asarray(triplets, dtype=int)
    edge12_ids = np.asarray(edge12_ids)
    edge23_ids = np.asarray(edge23_ids)
    edge_roles = _normalise_overlap_edge_roles(edge_roles)
    cell_positions = _normalise_overlap_cell_positions(cell_positions)

    edge_units = set()
    if "upstream" in edge_roles:
        edge_units.update(int(e) for e in edge12_ids[np.isfinite(edge12_ids.astype(float))])
    if "downstream" in edge_roles:
        edge_units.update(int(e) for e in edge23_ids[np.isfinite(edge23_ids.astype(float))])

    cell_units = set()
    if triplets.size > 0 and len(cell_positions) > 0:
        cell_units.update(int(x) for x in triplets[:, list(cell_positions)].ravel())

    return edge_units, cell_units


def _filter_negative_triplets_against_positive_units(
    neg_triplets,
    neg_edge12_ids,
    neg_edge23_ids,
    pos_triplets,
    pos_edge12_ids,
    pos_edge23_ids,
    sample_index,
    sample_name,
    edge_roles=("upstream", "downstream"),
    cell_positions=(0, 1, 2),
):
    """Remove MI-up-negative triplets that reuse MI-up-positive edges or cells.

    This makes all downstream analyses consistent: edge-level features, cell-level
    features, single-unit plots, and sample-aggregated plots are all computed from
    the same filtered triplet set.
    """
    neg_triplets = np.asarray(neg_triplets, dtype=int)
    neg_edge12_ids = np.asarray(neg_edge12_ids, dtype=int)
    neg_edge23_ids = np.asarray(neg_edge23_ids, dtype=int)
    pos_triplets = np.asarray(pos_triplets, dtype=int)
    pos_edge12_ids = np.asarray(pos_edge12_ids, dtype=int)
    pos_edge23_ids = np.asarray(pos_edge23_ids, dtype=int)

    edge_roles = _normalise_overlap_edge_roles(edge_roles)
    cell_positions = _normalise_overlap_cell_positions(cell_positions)

    n_before = int(neg_triplets.shape[0])

    pos_edge_units, pos_cell_units = _triplet_unit_sets_for_overlap(
        triplets=pos_triplets,
        edge12_ids=pos_edge12_ids,
        edge23_ids=pos_edge23_ids,
        edge_roles=edge_roles,
        cell_positions=cell_positions,
    )

    if n_before == 0:
        keep_mask = np.zeros(0, dtype=bool)
        edge_overlap = np.zeros(0, dtype=bool)
        cell_overlap = np.zeros(0, dtype=bool)
    else:
        edge_overlap = np.zeros(n_before, dtype=bool)
        if "upstream" in edge_roles:
            edge_overlap |= np.asarray([int(e) in pos_edge_units for e in neg_edge12_ids], dtype=bool)
        if "downstream" in edge_roles:
            edge_overlap |= np.asarray([int(e) in pos_edge_units for e in neg_edge23_ids], dtype=bool)

        cell_overlap = np.zeros(n_before, dtype=bool)
        if len(cell_positions) > 0:
            for pos in cell_positions:
                cell_overlap |= np.asarray([int(c) in pos_cell_units for c in neg_triplets[:, pos]], dtype=bool)

        keep_mask = ~(edge_overlap | cell_overlap)

    n_removed_edge = int(np.sum(edge_overlap))
    n_removed_cell = int(np.sum(cell_overlap))
    n_removed_both = int(np.sum(edge_overlap & cell_overlap))
    n_removed_any = int(np.sum(~keep_mask))
    n_after = int(np.sum(keep_mask))

    record = {
        "sample_index": int(sample_index),
        "sample_name": str(sample_name),
        "negative_group": str(GROUP_UP_NEG) if "GROUP_UP_NEG" in globals() else "MIup_not_interacting",
        "positive_group": str(GROUP_UP_POS) if "GROUP_UP_POS" in globals() else "MIup_interacting",
        "exclude_overlap_edge_roles": ",".join(edge_roles),
        "exclude_overlap_cell_positions": ",".join(str(x) for x in cell_positions),
        "n_positive_triplets_reference": int(pos_triplets.shape[0]),
        "n_positive_edge_units_reference": int(len(pos_edge_units)),
        "n_positive_cell_units_reference": int(len(pos_cell_units)),
        "n_negative_triplets_before_overlap_exclusion": n_before,
        "n_negative_triplets_removed_edge_overlap": n_removed_edge,
        "n_negative_triplets_removed_cell_overlap": n_removed_cell,
        "n_negative_triplets_removed_both_edge_and_cell_overlap": n_removed_both,
        "n_negative_triplets_removed_any_overlap": n_removed_any,
        "n_negative_triplets_after_overlap_exclusion": n_after,
        "overlap_exclusion_fraction": float(n_removed_any / max(n_before, 1)),
    }

    if n_removed_any > 0:
        print(
            f"[Cross-group overlap exclusion] sample={sample_name}: "
            f"removed {n_removed_any}/{n_before} MI-up-negative triplets "
            f"({record['overlap_exclusion_fraction']:.2%}) because they reuse "
            "MI-up-positive edges or cells."
        )

    return (
        neg_triplets[keep_mask],
        neg_edge12_ids[keep_mask],
        neg_edge23_ids[keep_mask],
        record,
    )


def _post_overlap_check_between_groups(
    triplet_df,
    sample_index,
    sample_name,
    edge_roles=("upstream", "downstream"),
    cell_positions=(0, 1, 2),
):
    """Check remaining cross-group edge/cell overlap after filtering."""
    edge_roles = _normalise_overlap_edge_roles(edge_roles)
    cell_positions = _normalise_overlap_cell_positions(cell_positions)

    sub = triplet_df[triplet_df["sample_index"].eq(sample_index)].copy()
    pos = sub[sub["group"].astype(str).eq(str(GROUP_UP_POS))]
    neg = sub[sub["group"].astype(str).eq(str(GROUP_UP_NEG))]

    def _edges_from_df(df):
        out = set()
        if "upstream" in edge_roles and "edge12_id" in df:
            out.update(int(x) for x in pd.to_numeric(df["edge12_id"], errors="coerce").dropna().astype(int))
        if "downstream" in edge_roles and "edge23_id" in df:
            out.update(int(x) for x in pd.to_numeric(df["edge23_id"], errors="coerce").dropna().astype(int))
        return out

    def _cells_from_df(df):
        out = set()
        for pos_i in cell_positions:
            col = f"cell{pos_i + 1}"
            if col in df:
                out.update(int(x) for x in pd.to_numeric(df[col], errors="coerce").dropna().astype(int))
        return out

    pos_edges = _edges_from_df(pos)
    neg_edges = _edges_from_df(neg)
    pos_cells = _cells_from_df(pos)
    neg_cells = _cells_from_df(neg)

    overlap_edges = pos_edges & neg_edges
    overlap_cells = pos_cells & neg_cells

    return {
        "sample_index": int(sample_index),
        "sample_name": str(sample_name),
        "edge_roles_checked": ",".join(edge_roles),
        "cell_positions_checked": ",".join(str(x) for x in cell_positions),
        "n_pos_triplets": int(len(pos)),
        "n_neg_triplets": int(len(neg)),
        "n_pos_edge_units": int(len(pos_edges)),
        "n_neg_edge_units": int(len(neg_edges)),
        "n_overlap_edge_units_after_filter": int(len(overlap_edges)),
        "n_pos_cell_units": int(len(pos_cells)),
        "n_neg_cell_units": int(len(neg_cells)),
        "n_overlap_cell_units_after_filter": int(len(overlap_cells)),
    }


In [ ]:

# ==============================================================
# Build pathway specs and selected gene sets
# ==============================================================

# Score SPP1 on upstream edges and THBS on downstream edges.
pathway_specs = [
    PathwaySpec(
        name=UPSTREAM_PATHWAY_NAME,
        pathway_names=UPSTREAM_PATHWAY_NAMES,
        edge="upstream",
        source_position=0,
        target_position=1,
    ),
    PathwaySpec(
        name=DOWNSTREAM_PATHWAY_NAME,
        pathway_names=DOWNSTREAM_PATHWAY_NAMES,
        edge="downstream",
        source_position=1,
        target_position=2,
    ),
]

# CascadeDefinition is used only to prepare sample-level data and index the MI dimensions.
# Grouping is custom and only gates on MI-up.
cascade_def = CascadeDefinition(
    cell_types=cell_types_triplet,
    mi_first_index=MI_first_index,
    mi_second_index=MI_second_index,
    mi_threshold=MI_UP_THRESHOLD_HIGH,
    cell_type_obs_key="cell.types",
)

analysis_config = AnalysisConfig(
    expression_layer=None,
    score_on_normalized_expression=True,
    deduplicate_cells_within_group=DEDUPLICATE_UNITS_WITHIN_GROUP,
    exclude_overlap_mode="none",
)

# Load LR meta if it has not already been loaded.
if "lr_meta" not in globals():
    lr_meta = pd.read_csv(input_path(Path(run_dirs["run_dir"]) / "LR_meta_incellchatdb.csv"))

# The analyzer supplies pathway indices and global expression/LR maxima.
# Group construction and scoring below do not call its default run() method.
_pathway_index_analyzer = MICascadeAnalyzer(
    adata_list=adata_list,
    factor_envir_norm_list=Factor_envir_list,
    hyper_edge_adj_list=result["hyper_edge_adj_list"],
    spidernet_data_list=SpiderNet_data_pyg_list,
    lr_meta=lr_meta,
    cascade_definition=cascade_def,
    pathway_specs=pathway_specs,
    gene_program_specs=[],
    config=analysis_config,
)

# -----------------------------
# CAF genes scored on cell2
# -----------------------------
caf_modules = {
    "myCAF": ["ACTA2", "TAGLN", "MYL9", "TPM2", "CNN1", "CALD1", "COL1A1", "COL1A2"],
    "iCAF": ["IL6", "CXCL12", "CXCL14", "LIF", "CCL2", "PTGS2"],
    "apCAF": ["HLA-DRA", "HLA-DRB1", "CD74", "CIITA"],
    "meCAF": ["COL11A1", "THBS2", "MMP11", "ITGA11", "FN1", "VCAN", "SPARC", "SULF1", "LOX", "PLOD2"],
    "periCAF": ["RGS5", "PDGFRB", "MCAM", "NOTCH3", "TAGLN"],
    "prolCAF": ["MKI67", "TOP2A", "PCNA"],
}
caf_all_markers = sorted(set(g for genes in caf_modules.values() for g in genes))
cell2_caf_genesets = {"CAF_all": caf_all_markers}

# -----------------------------
# Cell3 selected malignant gene sets
# -----------------------------
def _normalize_term(x):
    x = str(x).strip()
    x = re.sub(r"\s*\([^)]*\)\s*$", "", x)
    x = re.sub(r"\s+", " ", x)
    return x.lower()


def _gene_universe_from_adata_list():
    genes = []
    for adata in adata_list:
        genes.extend([str(g) for g in adata.var_names])
    return pd.Index(pd.unique(pd.Series(genes).astype(str)))


def _intersect_genes_with_universe(genes, gene_universe):
    upper_to_actual = {}
    for g in gene_universe.astype(str):
        upper_to_actual.setdefault(str(g).upper(), str(g))

    out = []
    seen = set()
    for g in genes:
        key = str(g).strip().upper()
        if key in upper_to_actual:
            actual = upper_to_actual[key]
            if actual not in seen:
                out.append(actual)
                seen.add(actual)
    return out


gene_universe = _gene_universe_from_adata_list()

selected_gene_sets = {}

# Hypoxia from CancerSEA_OV.
func_path = Path(DATA_ROOT) / "CancerSEA_OV" / "functional_geneset_list_df.csv"
func_df = pd.read_csv(input_path(func_path))
hypoxia_rows = func_df[func_df["GeneSet"].astype(str).map(_normalize_term).eq(_normalize_term("Hypoxia"))]
if hypoxia_rows.empty:
    raise KeyError(f"Cannot find Hypoxia in {func_path}")
selected_gene_sets["Hypoxia"] = _intersect_genes_with_universe(
    hypoxia_rows["Gene"].dropna().astype(str).tolist(),
    gene_universe,
)

# Prefer the KEGG reference CSV from HGSOC_Malignantsubtype_analysis_V2.
KEGG_PATHWAY_TO_DISPLAY = {
    "PD-L1 expression and PD-1 checkpoint pathway in cancer": "PD-1/PD-L1 checkpoint",
    "ECM-receptor interaction": "ECM-receptor interaction",
    "HIF-1 signaling pathway": "HIF-1 signaling",
}

_kegg_candidates = [
    Path(run_dirs["run_dir"]) / "TCGA_OV_KEGG_gene_signature" / "C5_four_KEGG_pathway_reference_genes_long.csv",
    Path(run_dirs["run_dir"]) / "C5_four_KEGG_pathway_reference_genes_long.csv",
    Path(DATA_ROOT) / "TCGA_OV_KEGG_gene_signature" / "C5_four_KEGG_pathway_reference_genes_long.csv",
]
_kegg_path = next((p for p in _kegg_candidates if input_path(p).exists()), None)

if _kegg_path is not None:
    kegg_df = pd.read_csv(input_path(_kegg_path))
    pathway_col = next(
        (c for c in ["Requested_Pathway", "Pathway", "GeneSet", "Library_Pathway"] if c in kegg_df.columns),
        None,
    )
    if pathway_col is None or "Gene" not in kegg_df.columns:
        raise KeyError(
            f"Cannot read KEGG pathway/gene columns from {_kegg_path}. "
            f"Available columns: {list(kegg_df.columns)}"
        )

    for requested_term, display_name in KEGG_PATHWAY_TO_DISPLAY.items():
        term_mask = kegg_df[pathway_col].astype(str).map(_normalize_term).eq(_normalize_term(requested_term))
        if not term_mask.any():
            available = sorted(kegg_df[pathway_col].astype(str).unique())[:20]
            raise KeyError(
                f"Cannot find pathway {requested_term!r} in {_kegg_path}. "
                f"First available terms: {available}"
            )

        selected_gene_sets[display_name] = _intersect_genes_with_universe(
            kegg_df.loc[term_mask, "Gene"].dropna().astype(str).tolist(),
            gene_universe,
        )
    kegg_source = str(_kegg_path)

else:
    # If the reference CSV is absent, use the predefined lists below.
    # Their membership may differ from the exported KEGG reference.
    fallback_kegg = {
        "PD-1/PD-L1 checkpoint": [
            "PDCD1", "CD274", "PDCD1LG2", "CD80", "CD86", "HLA-A", "HLA-B", "HLA-C",
            "JAK1", "JAK2", "STAT1", "STAT3", "IFNG", "CD8A", "CD8B", "PTEN", "PIK3CA",
        ],
        "ECM-receptor interaction": [
            "COL1A1", "COL1A2", "COL4A1", "COL4A2", "COL6A1", "COL6A2", "FN1",
            "LAMA3", "LAMB3", "LAMC2", "ITGA1", "ITGA2", "ITGA3", "ITGA5",
            "ITGA6", "ITGB1", "CD44", "THBS1", "THBS2", "SPP1",
        ],
        "HIF-1 signaling": [
            "HIF1A", "VEGFA", "SLC2A1", "HK1", "HK2", "LDHA", "PGK1", "ENO1",
            "PDK1", "EGLN1", "EGLN3", "CA9", "SERPINE1", "TFRC", "ANGPT2",
        ],
    }
    for display_name, genes in fallback_kegg.items():
        selected_gene_sets[display_name] = _intersect_genes_with_universe(genes, gene_universe)
    kegg_source = "fallback_hardcoded_gene_lists"

selected_gene_sets = {
    "PD-1/PD-L1 checkpoint": selected_gene_sets["PD-1/PD-L1 checkpoint"],
    "ECM-receptor interaction": selected_gene_sets["ECM-receptor interaction"],
    "HIF-1 signaling": selected_gene_sets["HIF-1 signaling"],
    "Hypoxia": selected_gene_sets["Hypoxia"],
}

# CAF_all lists the requested marker union; per-sample scoring matches genes
# to each AnnData object and records the number actually used.
gene_set_summary = []
for category, gene_sets in [
    ("Cell2_CAF", cell2_caf_genesets),
    ("Cell3_Selected", selected_gene_sets),
]:
    for name, genes in gene_sets.items():
        gene_set_summary.append({
            "category": category,
            "program": name,
            "n_genes_overlap_with_data": len(genes),
            "genes_used": ", ".join(genes),
        })

gene_set_summary_df = pd.DataFrame(gene_set_summary)
gene_set_summary_path = Path(output_dir) / f"{cascade_output_suffix}_gene_sets_used.csv"
gene_set_summary_df.to_csv(gene_set_summary_path, index=False)

print(f"KEGG gene source: {kegg_source}")
print(f"Saved gene-set summary: {gene_set_summary_path}")
display(gene_set_summary_df[["category", "program", "n_genes_overlap_with_data"]])


In [ ]:

# ==============================================================
# Extract upstream-gated triplets and compute all four feature classes
# --------------------------------------------------------------
# Features:
#   (1) MI-up and MI-down interaction strength
#   (2) pathway-level LR coexpression at upstream and downstream edges
#   (3) cell2 CAF score
#   (4) cell3 PD-1/PD-L1, ECM-receptor, HIF-1, Hypoxia module scores
#
# Important cross-group rule:
#   If EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS=True, MI-up-negative triplets are
#   removed when they reuse any MI-up-positive edge/cell within the same sample.
#   This makes triplet tables, single-unit features, and sample-level aggregates
#   all based on the same non-overlapping negative group.
# ==============================================================

feature_records = []
triplet_records = []
edge_state_records = []
edge_validation_records = []
overlap_exclusion_records = []

for sample_index, (adata, factor_arr, hyper_adj, sp_data) in enumerate(
    zip(
        adata_list,
        Factor_envir_list,
        result["hyper_edge_adj_list"],
        SpiderNet_data_pyg_list,
    )
):
    sample_name = _sample_name_from_adata(adata, sample_index)

    sample = spna.build_sample_prepared_data(
        adata=adata,
        factor_envir_norm=factor_arr,
        hyper_edge_adj=hyper_adj,
        spidernet_data=sp_data,
        cascade_definition=cascade_def,
        expression_max=_pathway_index_analyzer.expression_max,
        lr_max=_pathway_index_analyzer.lr_max,
        config=analysis_config,
    )

    # Use sample.edge_index as the source of truth for edge ids, because
    # factor_matrix and lr_matrix are aligned to the row order of edge_index.
    # edge_lookup from SpiderNet can be useful elsewhere, but it may drop
    # duplicate directed pairs and should not define n_edges here.
    directed_edge_lookup, duplicate_directed_edge_pairs = _build_directed_edge_lookup_from_edge_index(sample.edge_index)
    n_edges = _get_edge_count(sample.edge_index)
    factor_matrix = _prepare_factor_matrix_for_edges(factor_arr, n_edges=n_edges)

    up_edge_strength = factor_matrix[:, MI_first_index]
    down_edge_strength = factor_matrix[:, MI_second_index]

    cond_up_pos = up_edge_strength >= MI_UP_THRESHOLD_HIGH
    cond_up_neg = up_edge_strength < MI_UP_THRESHOLD_LOW
    cond_down_any = np.ones(n_edges, dtype=bool)

    edge_state_records.append({
        "sample_index": sample_index,
        "sample_name": sample_name,
        "n_edges_total": int(n_edges),
        "n_up_pos_edges": int(np.sum(cond_up_pos)),
        "n_up_neg_edges": int(np.sum(cond_up_neg)),
        "n_up_candidate_edges": int(np.sum((up_edge_strength >= MI_UP_THRESHOLD_LOW) & (up_edge_strength < MI_UP_THRESHOLD_HIGH))),
        "n_down_spatial_edges": int(np.sum(cond_down_any)),
        "n_duplicate_directed_edge_pairs": int(len(duplicate_directed_edge_pairs)),
    })

    # ----------------------------------------------------------
    # First extract and edge-validate BOTH groups.
    # We do not create records until after the optional overlap exclusion,
    # so all downstream analyses use the same final triplet sets.
    # ----------------------------------------------------------
    group_payloads = {}

    group_to_cond_first = {
        GROUP_UP_POS: cond_up_pos,
        GROUP_UP_NEG: cond_up_neg,
    }

    for group, cond_first in group_to_cond_first.items():
        triplets = spna.extract_triplets(
            hyper_edge_adj=sample.hyper_edge_adj,
            edge_index=sample.edge_index,
            cell_types=sample.cell_types,
            cond_first=cond_first,
            cond_second=cond_down_any,
            expected_cell_types=cell_types_triplet,
        )

        # Apply the optional C5 restriction only when enabled for malignant cell3.
        triplets = _filter_triplets_to_malignant_c5_if_needed(
            triplets=triplets,
            adata=adata,
            sample=sample,
            cascade_definition=cascade_def,
            cell3_position=2,
            group=group,
            context="upstream_gated_analysis",
        )

        triplets = np.asarray(triplets, dtype=int)

        if triplets.size == 0:
            group_payloads[group] = {
                "triplets": np.empty((0, 3), dtype=int),
                "edge12_ids": np.empty(0, dtype=int),
                "edge23_ids": np.empty(0, dtype=int),
            }
            edge_validation_records.append({
                "sample_index": int(sample_index),
                "sample_name": str(sample_name),
                "group": str(group),
                "n_triplets_before_edge_validation": 0,
                "n_missing_edge12": 0,
                "n_missing_edge23": 0,
                "n_missing_either_edge": 0,
                "n_triplets_after_edge_validation": 0,
                "edge_mapping_drop_fraction": 0.0,
                "max_allowed_drop_fraction": float(MAX_TRIPLET_EDGE_DROP_FRACTION),
            })
            continue

        # Map every triplet to its two directed spatial edges using sample.edge_index.
        # This preserves one-to-one alignment between triplets and edge ids.
        edge12_ids = _triplets_to_directed_edge_ids(
            triplets=triplets,
            edge_index=sample.edge_index,
            edge="upstream",
            directed_edge_lookup=directed_edge_lookup,
        )
        edge23_ids = _triplets_to_directed_edge_ids(
            triplets=triplets,
            edge_index=sample.edge_index,
            edge="downstream",
            directed_edge_lookup=directed_edge_lookup,
        )

        # Enforce that every analyzed triplet truly has both spatial edges:
        # cell1->cell2 and cell2->cell3.  A small invalid fraction is dropped;
        # a large invalid fraction stops the notebook as a direction/lookup error.
        triplets, edge12_ids, edge23_ids, validation_record = _validate_and_filter_triplet_edges(
            triplets=triplets,
            edge12_ids=edge12_ids,
            edge23_ids=edge23_ids,
            sample_index=sample_index,
            sample_name=sample_name,
            group=group,
            max_drop_fraction=MAX_TRIPLET_EDGE_DROP_FRACTION,
            raise_on_high_dropout=RAISE_ON_HIGH_EDGE_MAPPING_DROPOUT,
        )
        edge_validation_records.append(validation_record)

        group_payloads[group] = {
            "triplets": triplets,
            "edge12_ids": edge12_ids.astype(int),
            "edge23_ids": edge23_ids.astype(int),
        }

    # ----------------------------------------------------------
    # Optional cross-group overlap exclusion.
    # Remove MI-up-negative triplets that reuse MI-up-positive edges/cells.
    # ----------------------------------------------------------
    if EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS:
        pos_payload = group_payloads.get(GROUP_UP_POS, None)
        neg_payload = group_payloads.get(GROUP_UP_NEG, None)

        if pos_payload is not None and neg_payload is not None:
            (
                neg_triplets_filtered,
                neg_edge12_filtered,
                neg_edge23_filtered,
                overlap_record,
            ) = _filter_negative_triplets_against_positive_units(
                neg_triplets=neg_payload["triplets"],
                neg_edge12_ids=neg_payload["edge12_ids"],
                neg_edge23_ids=neg_payload["edge23_ids"],
                pos_triplets=pos_payload["triplets"],
                pos_edge12_ids=pos_payload["edge12_ids"],
                pos_edge23_ids=pos_payload["edge23_ids"],
                sample_index=sample_index,
                sample_name=sample_name,
                edge_roles=EXCLUDE_OVERLAP_EDGE_ROLES,
                cell_positions=EXCLUDE_OVERLAP_CELL_POSITIONS,
            )

            group_payloads[GROUP_UP_NEG] = {
                "triplets": neg_triplets_filtered,
                "edge12_ids": neg_edge12_filtered.astype(int),
                "edge23_ids": neg_edge23_filtered.astype(int),
            }
            overlap_exclusion_records.append(overlap_record)
    else:
        overlap_exclusion_records.append({
            "sample_index": int(sample_index),
            "sample_name": str(sample_name),
            "negative_group": str(GROUP_UP_NEG),
            "positive_group": str(GROUP_UP_POS),
            "exclude_overlap_edge_roles": ",".join(_normalise_overlap_edge_roles(EXCLUDE_OVERLAP_EDGE_ROLES)),
            "exclude_overlap_cell_positions": ",".join(str(x) for x in _normalise_overlap_cell_positions(EXCLUDE_OVERLAP_CELL_POSITIONS)),
            "n_positive_triplets_reference": int(group_payloads.get(GROUP_UP_POS, {}).get("triplets", np.empty((0, 3))).shape[0]),
            "n_positive_edge_units_reference": np.nan,
            "n_positive_cell_units_reference": np.nan,
            "n_negative_triplets_before_overlap_exclusion": int(group_payloads.get(GROUP_UP_NEG, {}).get("triplets", np.empty((0, 3))).shape[0]),
            "n_negative_triplets_removed_edge_overlap": 0,
            "n_negative_triplets_removed_cell_overlap": 0,
            "n_negative_triplets_removed_both_edge_and_cell_overlap": 0,
            "n_negative_triplets_removed_any_overlap": 0,
            "n_negative_triplets_after_overlap_exclusion": int(group_payloads.get(GROUP_UP_NEG, {}).get("triplets", np.empty((0, 3))).shape[0]),
            "overlap_exclusion_fraction": 0.0,
        })

    # ----------------------------------------------------------
    # Create records from the final group payloads.
    # ----------------------------------------------------------
    for group in UPSTREAM_GATED_GROUPS:
        payload = group_payloads.get(group, None)
        if payload is None:
            continue

        triplets = np.asarray(payload["triplets"], dtype=int)
        edge12_ids = np.asarray(payload["edge12_ids"], dtype=int)
        edge23_ids = np.asarray(payload["edge23_ids"], dtype=int)

        if triplets.shape[0] == 0:
            continue

        mi_up_values = _values_by_edge_and_mi(factor_matrix, edge12_ids, MI_first_index)
        mi_down_values = _values_by_edge_and_mi(factor_matrix, edge23_ids, MI_second_index)

        # Triplet table.
        for t_i, trip in enumerate(triplets):
            c1, c2, c3 = [int(x) for x in trip[:3]]
            triplet_records.append({
                "sample_index": sample_index,
                "sample_name": sample_name,
                "group": group,
                "triplet_local_index": t_i,
                "cell1": c1,
                "cell2": c2,
                "cell3": c3,
                "cell1_id": str(sample.obs_names[c1]),
                "cell2_id": str(sample.obs_names[c2]),
                "cell3_id": str(sample.obs_names[c3]),
                "cell1_type": str(sample.cell_types[c1]),
                "cell2_type": str(sample.cell_types[c2]),
                "cell3_type": str(sample.cell_types[c3]),
                "edge12_id": int(edge12_ids[t_i]) if pd.notna(edge12_ids[t_i]) else np.nan,
                "edge23_id": int(edge23_ids[t_i]) if pd.notna(edge23_ids[t_i]) else np.nan,
                "MI_up_strength": float(mi_up_values[t_i]) if np.isfinite(mi_up_values[t_i]) else np.nan,
                "MI_down_strength": float(mi_down_values[t_i]) if np.isfinite(mi_down_values[t_i]) else np.nan,
            })

        # ------------------------------------------------------
        # (1) MI-up / MI-down interaction strength
        # ------------------------------------------------------
        for edge_label, edge_ids, values, feature_name in [
            ("upstream", edge12_ids, mi_up_values, f"MI-{MI_first_of_interest} upstream strength"),
            ("downstream", edge23_ids, mi_down_values, f"MI-{MI_second_of_interest} downstream strength"),
        ]:
            for unit_i, (edge_id, val) in enumerate(zip(edge_ids, values)):
                feature_records.append({
                    "sample_index": sample_index,
                    "sample_name": sample_name,
                    "group": group,
                    "feature_category": "MI strength",
                    "feature": feature_name,
                    "value": float(val) if np.isfinite(val) else np.nan,
                    "unit_type": "edge",
                    "unit_id": f"{sample_index}:edge:{int(edge_id)}" if pd.notna(edge_id) else f"{sample_index}:edge:NA:{unit_i}",
                    "edge_role": edge_label,
                })

        # ------------------------------------------------------
        # (2) Pathway-level LR coexpression at upstream/downstream
        # ------------------------------------------------------
        for pathway_spec in pathway_specs:
            index_info = _pathway_index_analyzer.pathway_index_info[pathway_spec.name]
            edge_ids = edge12_ids if pathway_spec.edge == "upstream" else edge23_ids
            lr_scores = spna.compute_mean_score_per_unit(
                matrix=sample.lr_matrix,
                row_indices=edge_ids,
                feature_indices=index_info.lr_pair_indices,
            )

            feature_name = f"{pathway_spec.name} LR coexpression"
            for unit_i, (edge_id, val) in enumerate(zip(edge_ids, lr_scores)):
                feature_records.append({
                    "sample_index": sample_index,
                    "sample_name": sample_name,
                    "group": group,
                    "feature_category": "Pathway LR coexpression",
                    "feature": feature_name,
                    "value": float(val) if pd.notna(val) else np.nan,
                    "unit_type": "edge",
                    "unit_id": f"{sample_index}:edge:{int(edge_id)}" if pd.notna(edge_id) else f"{sample_index}:edge:NA:{unit_i}",
                    "edge_role": pathway_spec.edge,
                    "pathway_name": pathway_spec.name,
                    "n_lr_pairs_used": int(len(index_info.lr_pair_indices)),
                })

        # ------------------------------------------------------
        # (3) Cell2 CAF score
        # ------------------------------------------------------
        cell2_nodes = triplets[:, 1].astype(int)
        if DEDUPLICATE_UNITS_WITHIN_GROUP:
            cell2_nodes = np.unique(cell2_nodes)

        # For MODULE_SCORE_METHOD='zscore_mean', ref_cell2 defines the within-slice
        # reference cells. For MODULE_SCORE_METHOD='zscore_mean_global',
        # global_reference_cell_type=ct2_cur makes the helper compute gene mean/std
        # from all ct2_cur cells concatenated across all slices.
        ref_cell2 = np.where(np.asarray(sample.cell_types).astype(str) == ct2_cur)[0]

        for program_name, genes in cell2_caf_genesets.items():
            gene_indices, used_genes, missing_genes = _gene_indices_from_names(adata.var_names, genes)
            scores = _module_score_for_nodes(
                expression=sample.expression,
                node_ids=cell2_nodes,
                gene_indices=gene_indices,
                method=MODULE_SCORE_METHOD,
                reference_nodes=ref_cell2,
                var_names=adata.var_names,
                gene_names=used_genes,
                addmodule_ctrl_size=ADD_MODULE_CTRL_SIZE,
                addmodule_random_state=ADD_MODULE_RANDOM_STATE,
                addmodule_use_raw=ADD_MODULE_USE_RAW,
                global_reference_cell_type=ct2_cur,
            )

            for node_id, val in zip(cell2_nodes, scores):
                feature_records.append({
                    "sample_index": sample_index,
                    "sample_name": sample_name,
                    "group": group,
                    "feature_category": "Cell2 CAF score",
                    "feature": program_name,
                    "value": float(val) if np.isfinite(val) else np.nan,
                    "unit_type": "cell",
                    "unit_id": f"{sample_index}:cell:{int(node_id)}",
                    "cell_position": 1,
                    "cell_index": int(node_id),
                    "cell_id": str(sample.obs_names[int(node_id)]),
                    "cell_type": str(sample.cell_types[int(node_id)]),
                    "score_method": MODULE_SCORE_METHOD,
                    "n_genes_used": int(len(gene_indices)),
                })

        # ------------------------------------------------------
        # (4) Cell3 selected malignant module scores
        # ------------------------------------------------------
        cell3_nodes = triplets[:, 2].astype(int)
        if DEDUPLICATE_UNITS_WITHIN_GROUP:
            cell3_nodes = np.unique(cell3_nodes)

        # For MODULE_SCORE_METHOD='zscore_mean', ref_cell3 defines the within-slice
        # reference cells. For MODULE_SCORE_METHOD='zscore_mean_global',
        # global_reference_cell_type=ct3_cur makes the helper compute gene mean/std
        # from all ct3_cur cells concatenated across all slices.
        ref_cell3 = np.where(np.asarray(sample.cell_types).astype(str) == ct3_cur)[0]

        for program_name, genes in selected_gene_sets.items():
            gene_indices, used_genes, missing_genes = _gene_indices_from_names(adata.var_names, genes)
            scores = _module_score_for_nodes(
                expression=sample.expression,
                node_ids=cell3_nodes,
                gene_indices=gene_indices,
                method=MODULE_SCORE_METHOD,
                reference_nodes=ref_cell3,
                var_names=adata.var_names,
                gene_names=used_genes,
                addmodule_ctrl_size=ADD_MODULE_CTRL_SIZE,
                addmodule_random_state=ADD_MODULE_RANDOM_STATE,
                addmodule_use_raw=ADD_MODULE_USE_RAW,
                global_reference_cell_type=ct3_cur,
            )

            for node_id, val in zip(cell3_nodes, scores):
                feature_records.append({
                    "sample_index": sample_index,
                    "sample_name": sample_name,
                    "group": group,
                    "feature_category": "Cell3 selected module score",
                    "feature": program_name,
                    "value": float(val) if np.isfinite(val) else np.nan,
                    "unit_type": "cell",
                    "unit_id": f"{sample_index}:cell:{int(node_id)}",
                    "cell_position": 2,
                    "cell_index": int(node_id),
                    "cell_id": str(sample.obs_names[int(node_id)]),
                    "cell_type": str(sample.cell_types[int(node_id)]),
                    "score_method": MODULE_SCORE_METHOD,
                    "n_genes_used": int(len(gene_indices)),
                })


triplet_table_upstream_gated = pd.DataFrame(triplet_records)
feature_records_upstream_gated = pd.DataFrame(feature_records)
edge_state_summary_upstream_gated = pd.DataFrame(edge_state_records)
edge_validation_summary_upstream_gated = pd.DataFrame(edge_validation_records)
overlap_exclusion_summary_upstream_gated = pd.DataFrame(overlap_exclusion_records)

# Post-filter overlap check on the final triplet table.
post_overlap_check_records = []
if not triplet_table_upstream_gated.empty:
    for sample_index, sample_name in (
        triplet_table_upstream_gated[["sample_index", "sample_name"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    ):
        post_overlap_check_records.append(
            _post_overlap_check_between_groups(
                triplet_df=triplet_table_upstream_gated,
                sample_index=sample_index,
                sample_name=sample_name,
                edge_roles=EXCLUDE_OVERLAP_EDGE_ROLES,
                cell_positions=EXCLUDE_OVERLAP_CELL_POSITIONS,
            )
        )
post_overlap_check_summary_upstream_gated = pd.DataFrame(post_overlap_check_records)

# Remove repeated edge/cell units within sample × group × feature if requested.
if DEDUPLICATE_UNITS_WITHIN_GROUP and not feature_records_upstream_gated.empty:
    before = len(feature_records_upstream_gated)
    feature_records_upstream_gated = (
        feature_records_upstream_gated
        .sort_values(["sample_index", "group", "feature_category", "feature", "unit_id"])
        .drop_duplicates(["sample_index", "group", "feature_category", "feature", "unit_id"], keep="first")
        .reset_index(drop=True)
    )
    after = len(feature_records_upstream_gated)
    print(f"Deduplicated feature units: {before:,} -> {after:,}")

# Save outputs.
triplet_table_path = Path(output_dir) / f"{cascade_output_suffix}_triplet_table.csv"
feature_records_path = Path(output_dir) / f"{cascade_output_suffix}_single_unit_feature_records.csv"
edge_state_summary_path = Path(output_dir) / f"{cascade_output_suffix}_edge_state_summary.csv"
edge_validation_summary_path = Path(output_dir) / f"{cascade_output_suffix}_edge_validation_summary.csv"
overlap_exclusion_summary_path = Path(output_dir) / f"{cascade_output_suffix}_overlap_exclusion_summary.csv"
post_overlap_check_summary_path = Path(output_dir) / f"{cascade_output_suffix}_post_overlap_check_summary.csv"

triplet_table_upstream_gated.to_csv(triplet_table_path, index=False)
feature_records_upstream_gated.to_csv(feature_records_path, index=False)
edge_state_summary_upstream_gated.to_csv(edge_state_summary_path, index=False)
edge_validation_summary_upstream_gated.to_csv(edge_validation_summary_path, index=False)
overlap_exclusion_summary_upstream_gated.to_csv(overlap_exclusion_summary_path, index=False)
post_overlap_check_summary_upstream_gated.to_csv(post_overlap_check_summary_path, index=False)

print(f"Saved triplet table: {triplet_table_path}")
print(f"Saved single-unit feature records: {feature_records_path}")
print(f"Saved edge-state summary: {edge_state_summary_path}")
print(f"Saved edge-validation summary: {edge_validation_summary_path}")
print(f"Saved overlap-exclusion summary: {overlap_exclusion_summary_path}")
print(f"Saved post-overlap check summary: {post_overlap_check_summary_path}")

print("\nEdge-validation summary:")
if edge_validation_summary_upstream_gated.empty:
    print("No triplets reached edge validation.")
else:
    display(edge_validation_summary_upstream_gated)

print("\nCross-group overlap exclusion summary:")
if overlap_exclusion_summary_upstream_gated.empty:
    print("No overlap exclusion records.")
else:
    display(overlap_exclusion_summary_upstream_gated)

print("\nPost-filter overlap check:")
if post_overlap_check_summary_upstream_gated.empty:
    print("No post-filter overlap check records.")
else:
    display(post_overlap_check_summary_upstream_gated)

# Stop early if overlap exclusion is enabled but overlap remains.
if (
    EXCLUDE_MIUP_NEG_OVERLAP_WITH_POS
    and not post_overlap_check_summary_upstream_gated.empty
    and (
        (post_overlap_check_summary_upstream_gated["n_overlap_edge_units_after_filter"] > 0).any()
        or (post_overlap_check_summary_upstream_gated["n_overlap_cell_units_after_filter"] > 0).any()
    )
):
    raise ValueError(
        "Cross-group overlap remains after filtering. Check EXCLUDE_OVERLAP_EDGE_ROLES, "
        "EXCLUDE_OVERLAP_CELL_POSITIONS, and the triplet construction logic."
    )

print("\nTriplet counts:")
display(
    triplet_table_upstream_gated
    .groupby(["group"], observed=True)
    .size()
    .reindex(UPSTREAM_GATED_GROUPS)
    .rename("n_triplets")
    .reset_index()
)

print("\nFeature record counts:")
display(
    feature_records_upstream_gated
    .groupby(["feature_category", "feature", "group"], observed=True)
    .size()
    .rename("n_units")
    .reset_index()
)


In [ ]:
# ==============================================================
# Count unique MI-12-high -> MI-10-high triplets in every slice.
# Monocyte --(MI-12 >= 0.5)--> Fibroblast --(MI-10 >= 0.5)--> Malignant
# This additional downstream gate is used for counts, not feature comparisons.
# ==============================================================
CASCADE_COUNT_HIGH_THRESHOLD = 0.5

expected_cascade = {
    "MI_first": 12,
    "MI_second": 10,
    "celltype_triple": "Monocyte -> Fibroblast -> Malignant",
}
if (
    MI_first_of_interest != expected_cascade["MI_first"]
    or MI_second_of_interest != expected_cascade["MI_second"]
    or celltype_triple_of_interest != expected_cascade["celltype_triple"]
):
    raise ValueError(
        "This count plot is defined for MI-12 -> MI-10 and "
        "Monocyte -> Fibroblast -> Malignant. Re-run the settings and "
        "triplet-extraction cells with those values first."
    )
if not np.isclose(float(MI_UP_THRESHOLD_HIGH), CASCADE_COUNT_HIGH_THRESHOLD):
    raise ValueError(
        f"MI_UP_THRESHOLD_HIGH must be {CASCADE_COUNT_HIGH_THRESHOLD} before "
        "building triplet_table_upstream_gated for this analysis."
    )

required_count_columns = {
    "sample_index", "sample_name", "group",
    "cell1", "cell2", "cell3",
    "cell1_type", "cell2_type", "cell3_type",
    "MI_up_strength", "MI_down_strength",
}
missing_count_columns = required_count_columns - set(triplet_table_upstream_gated.columns)
if missing_count_columns:
    raise ValueError(
        f"triplet_table_upstream_gated is missing columns: {sorted(missing_count_columns)}"
    )

cascade_high_triplets = triplet_table_upstream_gated.loc[
    triplet_table_upstream_gated["group"].eq(GROUP_UP_POS)
    & triplet_table_upstream_gated["cell1_type"].eq("Monocyte")
    & triplet_table_upstream_gated["cell2_type"].eq("Fibroblast")
    & triplet_table_upstream_gated["cell3_type"].eq("Malignant")
    & (triplet_table_upstream_gated["MI_up_strength"] >= CASCADE_COUNT_HIGH_THRESHOLD)
    & (triplet_table_upstream_gated["MI_down_strength"] >= CASCADE_COUNT_HIGH_THRESHOLD)
].copy()

# Count each cell1-cell2-cell3 cascade occurrence once within each slice.
cascade_high_triplets = cascade_high_triplets.drop_duplicates(
    ["sample_index", "cell1", "cell2", "cell3"]
)

all_slice_rows = [
    {
        "sample_index": sample_index,
        "slice_id": _sample_name_from_adata(adata, sample_index),
    }
    for sample_index, adata in enumerate(adata_list)
]
cascade_count_by_slice = pd.DataFrame(all_slice_rows)
observed_cascade_counts = (
    cascade_high_triplets
    .groupby("sample_index", observed=True)
    .size()
    .rename("cascade_count")
    .reset_index()
)
cascade_count_by_slice = cascade_count_by_slice.merge(
    observed_cascade_counts, on="sample_index", how="left"
)
cascade_count_by_slice["cascade_count"] = (
    cascade_count_by_slice["cascade_count"].fillna(0).astype(int)
)
cascade_count_by_slice = cascade_count_by_slice.sort_values(
    ["cascade_count", "slice_id"], ascending=[False, True]
).reset_index(drop=True)

print(
    f"MI-12-high -> MI-10-high cascade counts by slice "
    f"(threshold >= {CASCADE_COUNT_HIGH_THRESHOLD}):"
)
display(cascade_count_by_slice)

fig_height = max(4.0, 0.22 * len(cascade_count_by_slice))
fig, ax = plt.subplots(figsize=(5.0, fig_height))
ax.barh(
    cascade_count_by_slice["slice_id"].astype(str),
    cascade_count_by_slice["cascade_count"],
    color="#A9D48E",
    edgecolor="none",
)
ax.set_xlabel("Cascade count (log scale)")
ax.set_ylabel("Slice ID")
ax.set_title(
    "Monocyte --(MI-12 high)--> Fibroblast --(MI-10 high)--> Malignant\n"
    f"MI high threshold = {CASCADE_COUNT_HIGH_THRESHOLD}"
)
ax.set_xscale("log")
positive_counts = cascade_count_by_slice.loc[
    cascade_count_by_slice["cascade_count"] > 0, "cascade_count"
]
if not positive_counts.empty:
    ax.set_xlim(0.8, max(float(positive_counts.max()) * 1.15, 1.2))
else:
    ax.set_xlim(0.8, 1.2)
ax.invert_yaxis()
for tick_label, count in zip(
    ax.get_yticklabels(), cascade_count_by_slice["cascade_count"].to_numpy()
):
    tick_label.set_color("#939598" if count == 0 else "black")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linewidth=0.4, alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()

cascade_count_output_stem = Path(output_dir) / (
    "MI12high_MI10high_Monocyte-Fibroblast-Malignant_cascade_count_by_slice"
)
cascade_count_by_slice.to_csv(
    str(cascade_count_output_stem) + ".csv", index=False
)
fig.savefig(str(cascade_count_output_stem) + ".png", dpi=300, bbox_inches="tight")
fig.savefig(str(cascade_count_output_stem) + ".pdf", bbox_inches="tight")
plt.show()
print("Saved:", str(cascade_count_output_stem) + ".csv")
print("Saved:", str(cascade_count_output_stem) + ".png")
print("Saved:", str(cascade_count_output_stem) + ".pdf")


In [ ]:
cascade_count_output_stem

In [ ]:
# ==============================================================
# In situ MI-12 -> MI-10 cascade in sample SMI_T10_F18 (SMI_T10_F018).
# Run the preceding triplet-extraction and high-high count cells first.
# Reuse their unique triplets with both MI activities >= 0.5.
# ==============================================================
from IPython.display import Image, display
from SpiderNet.analysis import safe_filename
from SpiderNet.visualization import insituplot_MIcascade

INSITU_SAMPLE = "SMI_T10_F18"

# The processed data use three digits for the field-of-view identifier.
insitu_sample_alias = re.sub(
    r"_F(\d+)$", lambda match: f"_F{int(match.group(1)):03d}", INSITU_SAMPLE
)
insitu_matches = [
    index for index, name in enumerate(samples_name_list)
    if str(name) in {INSITU_SAMPLE, insitu_sample_alias}
]
if len(insitu_matches) != 1:
    raise ValueError(
        f"Expected one sample matching {INSITU_SAMPLE!r} or "
        f"{insitu_sample_alias!r}; found {len(insitu_matches)}."
    )
insitu_sample_index = insitu_matches[0]
insitu_sample_name = str(samples_name_list[insitu_sample_index])
insitu_triplets = cascade_high_triplets.loc[
    cascade_high_triplets["sample_index"].eq(insitu_sample_index)
].copy()
if insitu_triplets.empty:
    raise ValueError(f"No high-high cascade triplets found for {insitu_sample_name}.")
if not insitu_triplets["sample_name"].eq(insitu_sample_name).all():
    raise ValueError("The triplet sample names do not match samples_name_list.")

insitu_adata = adata_list[insitu_sample_index]
insitu_spatial = np.asarray(insitu_adata.obsm["spatial"])
insitu_celltypes = insitu_adata.obs["cell.types"].astype(str).to_numpy()
if (
    insitu_spatial.shape != (insitu_adata.n_obs, 2)
    or not np.isfinite(insitu_spatial).all()
):
    raise ValueError("Expected finite, two-dimensional spatial coordinates per cell.")

# Confirm that cached node indices address the same cells as the spatial data.
insitu_nodes = insitu_triplets[["cell1", "cell2", "cell3"]].to_numpy(dtype=int)
if insitu_nodes.min() < 0 or insitu_nodes.max() >= insitu_adata.n_obs:
    raise ValueError("Cascade node indices are outside this sample's cell range.")
for insitu_position in range(3):
    insitu_expected_ids = insitu_triplets[f"cell{insitu_position + 1}_id"].astype(str)
    if not np.array_equal(
        insitu_adata.obs_names[insitu_nodes[:, insitu_position]].astype(str),
        insitu_expected_ids.to_numpy(),
    ):
        raise ValueError("Cascade cell IDs and spatial cell ordering do not match.")

# Use fixed cell-type colors even when a sample lacks one of the broad types.
insitu_color_map = {
    "B.cell": "#1f77b4", "Endothelial": "#17becf",
    "Fibroblast": "#c49c94", "Malignant": "#b22222",
    "Mast.cell": "#9467bd", "Monocyte": "#ff7f0e", "TNK.cell": "#2ca02c",
}
insitu_output_dir = Path(output_dir) / "MI12_MI10_cascade_insitu"

# Preserve the existing helper's one-edge-per-triplet rendering convention.
# Isolate its style changes so subsequent notebook plots retain their settings.
with plt.rc_context():
    insituplot_MIcascade(
        spatial=insitu_spatial,
        celltypes=insitu_celltypes,
        edge_first=insitu_nodes[:, [0, 1]],
        edge_second=insitu_nodes[:, [1, 2]],
        color_map=insitu_color_map,
        sample_index=insitu_sample_index,
        ct1=ct1_cur,
        ct2=ct2_cur,
        ct3=ct3_subtype_cur,
        MI_first_id=MI_first_of_interest,
        MI_second_id=MI_second_of_interest,
        save_dir=str(insitu_output_dir),
    )

# The helper saves PNG/PDF and closes its figure; explicitly show the PNG inline.
insitu_triple_safe = safe_filename(f"{ct1_cur} -> {ct2_cur} -> {ct3_subtype_cur}")
insitu_png_path = insitu_output_dir / "Insitu_High_order_MI" / (
    f"Spatial_{insitu_sample_index}_{insitu_triple_safe}"
    f"_MI-{MI_first_of_interest}_MI-{MI_second_of_interest}.png"
)
print(
    f"Sample: {insitu_sample_name} | {insitu_adata.n_obs:,} cells | "
    f"{len(insitu_triplets):,} unique cascade triplets\n"
    f"Both MI activities >= {CASCADE_COUNT_HIGH_THRESHOLD:g}; "
    f"MI-{MI_first_of_interest} arrows: pink; MI-{MI_second_of_interest} arrows: purple."
)
display(Image(filename=str(insitu_png_path), width=1000))


### 9.3. Cell/edge-level feature comparison

Each feature unit is a directed spatial edge or a cell after the configured within-group deduplication. Values are pooled across samples and compared using two-sided Mann-Whitney U tests without multiple-testing correction. The figure reports significance stars and pooled-SD standardized mean differences (SMDs).


In [ ]:

# ==============================================================
# Single-cell / edge-unit level plots and statistics
# ==============================================================

single_unit_stats = []

for (feature_category, feature), sub in feature_records_upstream_gated.groupby(
    ["feature_category", "feature"],
    observed=True,
):
    x = sub.loc[sub["group"].eq(GROUP_UP_POS), "value"].to_numpy(dtype=float)
    y = sub.loc[sub["group"].eq(GROUP_UP_NEG), "value"].to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    pval = _compare_two_groups(x, y, paired=False, alternative="two-sided")

    single_unit_stats.append({
        "feature_category": feature_category,
        "feature": feature,
        "level": "single_unit",
        "test": "Mann-Whitney U",
        f"n_{GROUP_UP_POS}": int(len(x)),
        f"n_{GROUP_UP_NEG}": int(len(y)),
        f"mean_{GROUP_UP_POS}": float(np.nanmean(x)) if len(x) else np.nan,
        f"mean_{GROUP_UP_NEG}": float(np.nanmean(y)) if len(y) else np.nan,
        f"median_{GROUP_UP_POS}": float(np.nanmedian(x)) if len(x) else np.nan,
        f"median_{GROUP_UP_NEG}": float(np.nanmedian(y)) if len(y) else np.nan,
        f"mean_diff_{GROUP_UP_POS}_minus_{GROUP_UP_NEG}": (
            float(np.nanmean(x) - np.nanmean(y)) if len(x) and len(y) else np.nan
        ),
        "pvalue": pval,
        "pvalue_star": _p_to_star(pval),
    })

single_unit_stats_df = pd.DataFrame(single_unit_stats)
single_unit_stats_path = Path(output_dir) / f"{cascade_output_suffix}_single_unit_stats.csv"
single_unit_stats_df.to_csv(single_unit_stats_path, index=False)
print(f"Saved single-unit statistics: {single_unit_stats_path}")
display(single_unit_stats_df)

# Plot all four feature classes in one organized faceted figure.
plot_feature_panels_with_samplesize(
    feature_records_upstream_gated,
    ylabel="Single-unit score",
    figure_title=(
        f"Upstream-gated comparison, single-cell/edge-unit level\n"
        f"{ct1_cur}->{ct2_cur}->{ct3_subtype_cur}; MI-up={MI_first_of_interest}, MI-down={MI_second_of_interest}"
    ),
    output_path=Path(output_dir) / f"{cascade_output_suffix}_single_unit_feature_boxplots",
    pvalue_alternative="two-sided",
    ncol=4,
    show=True,
)


### 9.4. Sample-level feature comparison

For each feature, average unit scores within each sample and group; also export medians and unit counts. With the current settings, the statistics CSV uses a two-sided paired Wilcoxon signed-rank test on samples containing both groups. No multiple-testing correction is applied. The plot includes those matched samples, but its helper computes unpaired Mann-Whitney U annotation stars independently of the CSV test.


In [ ]:

# ==============================================================
# Sample-level aggregation, statistics, and plots
# ==============================================================

sample_level_feature_df = (
    feature_records_upstream_gated
    .dropna(subset=["value"])
    .groupby(
        ["feature_category", "feature", "sample_index", "sample_name", "group"],
        as_index=False,
        observed=True,
    )
    .agg(
        sample_mean_score=("value", "mean"),
        sample_median_score=("value", "median"),
        n_units=("value", "size"),
    )
)

sample_level_feature_path = Path(output_dir) / f"{cascade_output_suffix}_sample_level_feature_scores.csv"
sample_level_feature_df.to_csv(sample_level_feature_path, index=False)
print(f"Saved sample-level feature scores: {sample_level_feature_path}")

sample_level_stats = []

for (feature_category, feature), sub in sample_level_feature_df.groupby(
    ["feature_category", "feature"],
    observed=True,
):
    wide = sub.pivot_table(
        index="sample_name",
        columns="group",
        values="sample_mean_score",
        aggfunc="mean",
    )

    x_all = sub.loc[sub["group"].eq(GROUP_UP_POS), "sample_mean_score"].to_numpy(dtype=float)
    y_all = sub.loc[sub["group"].eq(GROUP_UP_NEG), "sample_mean_score"].to_numpy(dtype=float)
    x_all = x_all[np.isfinite(x_all)]
    y_all = y_all[np.isfinite(y_all)]

    if USE_PAIRED_SAMPLE_TEST:
        if (GROUP_UP_POS in wide.columns) and (GROUP_UP_NEG in wide.columns):
            paired = wide[[GROUP_UP_POS, GROUP_UP_NEG]].dropna()
        else:
            paired = pd.DataFrame(columns=[GROUP_UP_POS, GROUP_UP_NEG])

        x = paired[GROUP_UP_POS].to_numpy(dtype=float) if GROUP_UP_POS in paired.columns else np.asarray([])
        y = paired[GROUP_UP_NEG].to_numpy(dtype=float) if GROUP_UP_NEG in paired.columns else np.asarray([])

        pval = _compare_two_groups(x, y, paired=True, alternative="two-sided")
        test_name = "paired Wilcoxon signed-rank"
        n_test = int(len(paired))
        mean_pos = float(np.nanmean(x)) if len(x) else np.nan
        mean_neg = float(np.nanmean(y)) if len(y) else np.nan
        median_pos = float(np.nanmedian(x)) if len(x) else np.nan
        median_neg = float(np.nanmedian(y)) if len(y) else np.nan
        mean_diff = float(np.nanmean(x - y)) if len(x) else np.nan
        median_diff = float(np.nanmedian(x - y)) if len(x) else np.nan

    else:
        pval = _compare_two_groups(x_all, y_all, paired=False, alternative="two-sided")
        test_name = "unpaired Mann-Whitney U"
        n_test = int(len(x_all) + len(y_all))
        mean_pos = float(np.nanmean(x_all)) if len(x_all) else np.nan
        mean_neg = float(np.nanmean(y_all)) if len(y_all) else np.nan
        median_pos = float(np.nanmedian(x_all)) if len(x_all) else np.nan
        median_neg = float(np.nanmedian(y_all)) if len(y_all) else np.nan
        mean_diff = float(mean_pos - mean_neg) if np.isfinite(mean_pos) and np.isfinite(mean_neg) else np.nan
        median_diff = float(median_pos - median_neg) if np.isfinite(median_pos) and np.isfinite(median_neg) else np.nan

    sample_level_stats.append({
        "feature_category": feature_category,
        "feature": feature,
        "level": "sample_level",
        "test": test_name,
        "use_paired": bool(USE_PAIRED_SAMPLE_TEST),
        "n_test_samples_or_values": n_test,
        f"n_samples_{GROUP_UP_POS}": int(len(x_all)),
        f"n_samples_{GROUP_UP_NEG}": int(len(y_all)),
        f"mean_{GROUP_UP_POS}": mean_pos,
        f"mean_{GROUP_UP_NEG}": mean_neg,
        f"median_{GROUP_UP_POS}": median_pos,
        f"median_{GROUP_UP_NEG}": median_neg,
        f"mean_diff_{GROUP_UP_POS}_minus_{GROUP_UP_NEG}": mean_diff,
        f"median_diff_{GROUP_UP_POS}_minus_{GROUP_UP_NEG}": median_diff,
        "pvalue": pval,
        "pvalue_star": _p_to_star(pval),
    })

sample_level_stats_df = pd.DataFrame(sample_level_stats)
sample_level_stats_path = Path(output_dir) / f"{cascade_output_suffix}_sample_level_stats.csv"
sample_level_stats_df.to_csv(sample_level_stats_path, index=False)
print(f"Saved sample-level statistics: {sample_level_stats_path}")
display(sample_level_stats_df)

# Plotting DataFrame:
# In paired mode, include only samples with both groups for each feature.
# Plot annotations still use the unpaired test in the shared plotting helper.
plot_sample_df = sample_level_feature_df.rename(columns={"sample_mean_score": "value"}).copy()

if USE_PAIRED_SAMPLE_TEST:
    keep_parts = []
    for (feature_category, feature), sub in plot_sample_df.groupby(["feature_category", "feature"], observed=True):
        wide = sub.pivot_table(index="sample_name", columns="group", values="value", aggfunc="mean")
        if (GROUP_UP_POS not in wide.columns) or (GROUP_UP_NEG not in wide.columns):
            continue
        paired_samples = set(wide[[GROUP_UP_POS, GROUP_UP_NEG]].dropna().index.astype(str))
        keep_parts.append(sub[sub["sample_name"].astype(str).isin(paired_samples)])
    plot_sample_df = pd.concat(keep_parts, ignore_index=True) if keep_parts else plot_sample_df.iloc[0:0].copy()

plot_feature_panels_with_samplesize(
    plot_sample_df,
    ylabel="Sample-level mean score",
    figure_title=(
        f"Upstream-gated comparison, sample-aggregated level\n"
        f"{ct1_cur}->{ct2_cur}->{ct3_subtype_cur}; MI-up={MI_first_of_interest}, MI-down={MI_second_of_interest}; "
        f"{'paired' if USE_PAIRED_SAMPLE_TEST else 'unpaired'} test"
    ),
    output_path=Path(output_dir) / f"{cascade_output_suffix}_sample_level_feature_boxplots",
    pvalue_alternative="two-sided",
    ncol=4,
    show=True,
)
